# AURA — AI Research & Decision Agent

### Scientific & Technical Research MVP

**Goal:** Transform a scientific or technical research question into
verified evidence, technical recommendations, and an actionable
implementation roadmap.

AURA integrates multiple research sources:

- scientific papers
- datasets
- code repositories
- official technical documentation

The system preserves source provenance and distinguishes verified
evidence from engineering assumptions.

## Phase 1 — Setup and Configuration

This section initializes the Python environment, loads dependencies,
connects to the Gemini API, and prepares shared utilities used
throughout AURA.

In [1]:
!pip install -q google-genai pydantic requests beautifulsoup4 python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import time
import base64
import requests

from typing import List
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field

from google import genai

In [3]:
api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("✅ Gemini API key loaded.")
else:
    raise ValueError(
        "GEMINI_API_KEY was not found in environment variables."
    )

✅ Gemini API key loaded.


In [4]:
client = genai.Client(
    api_key=api_key
)

print("✅ Gemini client initialized.")

✅ Gemini client initialized.


## Phase 2 — Model Router

AURA uses different Gemini models for different levels of reasoning.

The router also provides fallback behavior when a model is temporarily
unavailable or its quota is exhausted.

This improves:

- reliability
- cost efficiency
- resilience to API failures

In [5]:
FAST_MODEL = "gemini-3.5-flash-lite"
DEFAULT_MODEL = "gemini-3.5-flash"
REASONING_MODEL = "gemini-3.8-flash"

print("Fast model:", FAST_MODEL)
print("Default model:", DEFAULT_MODEL)
print("Reasoning model:", REASONING_MODEL)

Fast model: gemini-3.5-flash-lite
Default model: gemini-3.5-flash
Reasoning model: gemini-3.8-flash


In [6]:
def generate_with_fallback(
    prompt: str,
    preferred_model: str = DEFAULT_MODEL
):
    models_to_try = [
        preferred_model,
        DEFAULT_MODEL,
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        FAST_MODEL
    ]

    # Remove duplicate model names while preserving order
    models_to_try = list(
        dict.fromkeys(models_to_try)
    )

    last_error = None

    for model_name in models_to_try:

        try:
            print(f"Trying model: {model_name}")

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            print(
                f"✅ Success with: {model_name}"
            )

            return response

        except Exception as e:
            last_error = e
            error_text = str(e).lower()

            recoverable_errors = [
                "503",
                "unavailable",
                "429",
                "resource_exhausted",
                "quota"
            ]

            if any(
                error in error_text
                for error in recoverable_errors
            ):
                print(
                    f"⚠️ {model_name} unavailable "
                    f"or quota exhausted. "
                    f"Trying another model..."
                )

                continue

            raise e

    raise RuntimeError(
        f"All Gemini models failed. "
        f"Last error: {last_error}"
    )

In [7]:
def clean_json_response(text: str):
    clean_text = text.strip()

    if clean_text.startswith("```json"):
        clean_text = clean_text[7:]

    elif clean_text.startswith("```"):
        clean_text = clean_text[3:]

    if clean_text.endswith("```"):
        clean_text = clean_text[:-3]

    return json.loads(
        clean_text.strip()
    )

## Phase 3 — Task Planner Agent

The Task Planner analyzes the user's research question and converts it
into a structured research plan.

Responsibilities:

- identify the main research goal
- identify the technical/scientific domain
- capture important constraints
- break the problem into research tasks
- generate search queries for papers, datasets, code, and documentation

In [8]:
class ResearchTask(BaseModel):
    id: int
    task_type: str
    query: str
    purpose: str


class ResearchPlan(BaseModel):
    goal: str
    domain: str
    constraints: List[str] = Field(default_factory=list)
    tasks: List[ResearchTask]

In [9]:
PLANNER_PROMPT = """
You are the Task Planner Agent of AURA,
an AI Research & Decision Agent.

Your job is to analyze a scientific or technical research question
and create a structured research plan.

You must:

1. Understand the user's main goal.
2. Identify the scientific or technical domain.
3. Identify important constraints if they exist.
4. Break the problem into clear research tasks.
5. Create useful search queries for each task.

Possible task types include:

- paper_search
- dataset_search
- code_search
- documentation_search
- method_comparison
- limitation_analysis
- implementation_planning

Do not answer the research question itself.

Your job is only to PLAN the research.
"""

In [10]:
def create_research_plan(user_question: str):
    prompt = f"""
{PLANNER_PROMPT}

USER RESEARCH QUESTION:

{user_question}

Return ONLY valid JSON using exactly this structure:

{{
    "goal": "main research goal",
    "domain": "scientific or technical domain",
    "constraints": [
        "constraint 1"
    ],
    "tasks": [
        {{
            "id": 1,
            "task_type": "paper_search",
            "query": "search query",
            "purpose": "why this research task is needed"
        }}
    ]
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    plan_dict = clean_json_response(
        response.text
    )

    return ResearchPlan(
        **plan_dict
    )

In [11]:
question = """
I want to build a solar flare prediction system.

Find relevant scientific papers, datasets, machine learning methods,
existing code implementations, and technical documentation.

Compare the approaches, identify their limitations,
and propose an implementation plan.
"""

In [12]:
plan = create_research_plan(
    question
)

print(
    json.dumps(
        plan.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "goal": "Build a comprehensive solar flare prediction system by analyzing relevant literature, datasets, machine learning methods, and code implementations, followed by method comparison, limitation analysis, and an implementation plan.",
  "domain": "Space Weather and Machine Learning",
  "constraints": [
    "Must utilize publicly available solar observation datasets (e.g., SOHO, SDO/HMI)",
    "Must evaluate machine learning and deep learning methodologies",
    "Must identify operational limitations such as false alarm rates and lead time constraints",
    "Must produce an actionable implementation plan"
  ],
  "task

## Phase 4 — Scientific Paper Discovery

This stage searches for real scientific papers related to the
research question.

Responsibilities:

- use the paper-search task generated by the planner
- retrieve real academic sources
- preserve paper metadata and provenance
- normalize source metadata for downstream evidence extraction

In [13]:
class PaperSource(BaseModel):
    title: str
    authors: List[str] = Field(default_factory=list)
    year: int | None = None
    abstract: str | None = None
    url: str | None = None
    citation_count: int | None = None
    paper_id: str | None = None

In [14]:
paper_tasks = [
    task
    for task in plan.tasks
    if task.task_type == "paper_search"
]

paper_query = paper_tasks[0].query

print("Paper search query:")
print(paper_query)

Paper search query:
solar flare prediction machine learning deep learning review state of the art


In [15]:
def search_openalex(
    query: str,
    limit: int = 5
):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": limit
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [16]:
raw_openalex = search_openalex(
    query=paper_query,
    limit=5
)

print(
    f"Papers returned: "
    f"{len(raw_openalex.get('results', []))}"
)

Papers returned: 5


In [17]:
def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return None

    positions = []

    for word, indexes in inverted_index.items():
        for index in indexes:
            positions.append((index, word))

    positions.sort(
        key=lambda item: item[0]
    )

    return " ".join(
        word
        for _, word in positions
    )

In [18]:
def normalize_openalex_papers(search_result):
    papers = []

    for work in search_result.get("results", []):
        authors = []

        for authorship in work.get("authorships", []):
            author = authorship.get("author", {})
            name = author.get("display_name")

            if name:
                authors.append(name)

        abstract = reconstruct_abstract(
            work.get("abstract_inverted_index")
        )

        paper = PaperSource(
            title=work.get("display_name", ""),
            authors=authors,
            year=work.get("publication_year"),
            abstract=abstract,
            url=work.get("doi") or work.get("id"),
            citation_count=work.get("cited_by_count"),
            paper_id=work.get("id")
        )

        papers.append(paper)

    return papers

In [19]:
papers = normalize_openalex_papers(
    raw_openalex
)

print(
    f"✅ {len(papers)} papers normalized."
)

✅ 5 papers normalized.


In [20]:
for index, paper in enumerate(
    papers,
    start=1
):
    print("=" * 90)
    print(f"PAPER #{index}")
    print(f"Title: {paper.title}")
    print(f"Year: {paper.year}")
    print(f"Citations: {paper.citation_count}")
    print(f"URL: {paper.url}")
    print()

PAPER #1
Title: A review on extreme learning machine
Year: 2021
Citations: 562
URL: https://doi.org/10.1007/s11042-021-11007-7

PAPER #2
Title: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Year: 2019
Citations: 415
URL: https://doi.org/10.1029/2018sw002061

PAPER #3
Title: Machine learning in solar physics
Year: 2023
Citations: 70
URL: https://doi.org/10.1007/s41116-023-00038-x

PAPER #4
Title: A Review on Deep Learning Techniques for IoT Data
Year: 2022
Citations: 132
URL: https://doi.org/10.3390/electronics11101604

PAPER #5
Title: The flare likelihood and region eruption forecasting (FLARECAST) project: flare forecasting in the big data & machine learning era
Year: 2021
Citations: 75
URL: https://doi.org/10.1051/swsc/2021023



## Phase 5 — Paper Relevance Evaluation

Retrieved papers are evaluated against the original research question
before expensive evidence extraction.

Responsibilities:

- identify directly relevant papers
- remove off-topic search results
- preserve supporting papers when useful
- reduce unnecessary LLM calls

In [21]:
class PaperRelevanceAssessment(BaseModel):
    paper_id: str | None = None
    source_title: str
    relevance: str
    keep: bool
    reason: str

In [22]:
def evaluate_paper_relevance(
    user_question: str,
    papers: list
):
    paper_payload = []

    for paper in papers:
        paper_payload.append({
            "paper_id": paper.paper_id,
            "title": paper.title,
            "year": paper.year,
            "abstract": (
                paper.abstract[:4000]
                if paper.abstract
                else None
            )
        })

    paper_text = json.dumps(
        paper_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Paper Relevance Evaluator of AURA.

USER RESEARCH QUESTION:

{user_question}

CANDIDATE PAPERS:

{paper_text}

Evaluate each paper for relevance to the user's
solar flare prediction research goal.

Rules:

1. relevance must be:
   - high
   - medium
   - low

2. keep=true for papers that should contribute to
   downstream scientific analysis.

3. Keep directly relevant solar flare forecasting papers.

4. General solar physics or space-weather machine learning papers
   may be medium or high if they provide useful scientific context.

5. Generic machine learning papers unrelated to solar physics
   should be low and keep=false.

6. Do not use outside knowledge.
7. Base the decision only on the supplied title and abstract.

Return ONLY valid JSON as an array:

[
    {{
        "paper_id": "...",
        "source_title": "...",
        "relevance": "high",
        "keep": true,
        "reason": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return [
        PaperRelevanceAssessment(**item)
        for item in result
    ]

In [23]:
paper_relevance_results = evaluate_paper_relevance(
    question,
    papers
)

for index, result in enumerate(
    paper_relevance_results,
    start=1
):
    print("=" * 90)
    print(f"PAPER #{index}")
    print(f"Title: {result.source_title}")
    print(f"Relevance: {result.relevance}")
    print(f"Keep: {result.keep}")
    print(f"Reason: {result.reason}")
    print()

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
PAPER #1
Title: A review on extreme learning machine
Relevance: low
Keep: False
Reason: This is a generic machine learning review paper focused on Extreme Learning Machines (ELM) and their applications in medical imaging, with no mention of solar physics or solar flares.

PAPER #2
Title: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Relevance: high
Keep: True
Reason: This paper directly discusses machine learning for space weather forecasting, specifically mentioning the forecasting of solar flares occurrence, and addresses open challenges and probabilistic approaches.

PAPER #3
Title: Machine learning in solar physics
Relevance: high
Keep: True
Reason: This paper provides direct context on the application of machine learning and deep learning to analyze solar observation data and improve the understanding and prediction of explosive events like solar flares.

PAPER #4
Title: A Re

In [24]:
accepted_paper_ids = {
    result.paper_id
    for result in paper_relevance_results
    if result.keep
}

relevant_papers = [
    paper
    for paper in papers
    if paper.paper_id in accepted_paper_ids
]

print(f"Retrieved papers: {len(papers)}")
print(f"Relevant papers: {len(relevant_papers)}")

Retrieved papers: 5
Relevant papers: 3


## Phase 6 — Paper Evidence Extraction

This stage extracts structured scientific evidence from relevant papers.

Responsibilities:

- extract evidence only from the supplied paper abstract
- identify research focus, methods, data sources, findings, and limitations
- preserve paper provenance
- avoid unsupported claims

In [25]:
class EvidenceItem(BaseModel):
    paper_id: str | None = None
    source_title: str
    source_url: str | None = None

    research_focus: str

    methods: List[str] = Field(default_factory=list)
    datasets_or_data_sources: List[str] = Field(default_factory=list)
    findings: List[str] = Field(default_factory=list)
    limitations: List[str] = Field(default_factory=list)

    evidence_summary: str

In [26]:
def extract_paper_evidence(
    paper: PaperSource
):
    if not paper.abstract:
        return None

    prompt = f"""
You are the Scientific Evidence Extraction Agent of AURA.

Extract structured scientific evidence ONLY from the supplied
paper title and abstract.

Do not use outside knowledge.
Do not invent methods, datasets, findings, or limitations.

If something is not explicitly supported by the abstract,
return an empty list for that field.

PAPER TITLE:

{paper.title}

ABSTRACT:

{paper.abstract}

Return ONLY valid JSON:

{{
    "paper_id": "{paper.paper_id}",
    "source_title": "{paper.title}",
    "source_url": "{paper.url}",
    "research_focus": "short description of the paper's research focus",
    "methods": [],
    "datasets_or_data_sources": [],
    "findings": [],
    "limitations": [],
    "evidence_summary": "short evidence-grounded summary"
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return EvidenceItem(**result)

In [27]:
paper_evidence_store = []

for index, paper in enumerate(
    relevant_papers,
    start=1
):
    print(
        f"Processing paper "
        f"{index}/{len(relevant_papers)}: "
        f"{paper.title}"
    )

    try:
        evidence = extract_paper_evidence(
            paper
        )

        if evidence:
            paper_evidence_store.append(
                evidence
            )

            print(
                "✅ Evidence extracted."
            )
        else:
            print(
                "⚠️ Skipped: "
                "No abstract available."
            )

    except Exception as e:
        print(
            f"❌ Extraction failed: {e}"
        )

    print()

print(
    f"✅ Paper evidence items: "
    f"{len(paper_evidence_store)}"
)

Processing paper 1/3: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Evidence extracted.

Processing paper 2/3: Machine learning in solar physics
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Evidence extracted.

Processing paper 3/3: The flare likelihood and region eruption forecasting (FLARECAST) project: flare forecasting in the big data & machine learning era
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Evidence extracted.

✅ Paper evidence items: 3


In [28]:
for index, evidence in enumerate(
    paper_evidence_store,
    start=1
):
    print("=" * 90)
    print(f"EVIDENCE #{index}")
    print(f"Source: {evidence.source_title}")
    print(f"Focus: {evidence.research_focus}")
    print(f"Methods: {evidence.methods}")
    print(
        f"Data sources: "
        f"{evidence.datasets_or_data_sources}"
    )
    print(f"Findings: {evidence.findings}")
    print(
        f"Limitations: "
        f"{evidence.limitations}"
    )
    print(
        f"Summary: "
        f"{evidence.evidence_summary}"
    )
    print()

EVIDENCE #1
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Focus: Reviewing the present and future role of machine learning in Space Weather nowcasting and forecasting, focusing on specific active areas and highlighting open challenges for the next decade.
Methods: ['machine learning', 'physics-based and machine learning approaches (gray box)', 'probabilistic approach focused on the reliable assessment of uncertainties']
Data sources: []
Findings: ['Machine learning has seen the most activity in forecasting geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flares occurrence, coronal mass ejection propagation time, and solar wind speed.']
Limitations: []
Summary: This Grand Challenge review paper discusses the application of machine learning in Space Weather nowcasting and forecasting. It highlights previous works focused on geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flares, coronal mass e

## Phase 7 — Claim Verification

This stage verifies extracted scientific claims against the original
paper abstracts.

Responsibilities:

- compare extracted claims with the original source abstract
- identify supported, partially supported, and unsupported claims
- prevent unsupported evidence from entering downstream analysis
- preserve source provenance

In [29]:
class ClaimVerification(BaseModel):
    claim: str
    status: str
    explanation: str


class PaperVerification(BaseModel):
    paper_id: str | None = None
    source_title: str
    verified_claims: List[ClaimVerification] = Field(
        default_factory=list
    )

In [30]:
paper_lookup = {
    paper.paper_id: paper
    for paper in relevant_papers
}

In [31]:
def build_claim_candidates(
    evidence: EvidenceItem
):
    claims = []

    claims.extend(evidence.findings)

    for method in evidence.methods:
        claims.append(
            f"The paper uses or discusses the method: {method}"
        )

    for data_source in evidence.datasets_or_data_sources:
        claims.append(
            f"The paper uses or discusses the data source: {data_source}"
        )

    if evidence.evidence_summary:
        claims.append(
            f"Evidence summary: {evidence.evidence_summary}"
        )

    # Remove duplicates while preserving order
    return list(dict.fromkeys(claims))

In [32]:
def verify_paper_evidence(
    evidence: EvidenceItem
):
    paper = paper_lookup.get(
        evidence.paper_id
    )

    if paper is None or not paper.abstract:
        return None

    claims = build_claim_candidates(
        evidence
    )

    claims_text = json.dumps(
        claims,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Scientific Claim Verification Agent of AURA.

Verify each extracted claim ONLY against the ORIGINAL PAPER ABSTRACT.

PAPER TITLE:

{paper.title}

ORIGINAL ABSTRACT:

{paper.abstract}

CLAIMS:

{claims_text}

For every claim assign exactly one status:

- supported
- partially_supported
- unsupported

Rules:

1. supported:
   The abstract directly supports the claim.

2. partially_supported:
   Only part of the claim is supported or the wording is broader
   than the abstract.

3. unsupported:
   The abstract does not justify the claim.

4. Do not use outside knowledge.
5. Be strict.
6. Do not infer missing information.

Return ONLY valid JSON:

{{
    "paper_id": "{paper.paper_id}",
    "source_title": "{paper.title}",
    "verified_claims": [
        {{
            "claim": "...",
            "status": "supported",
            "explanation": "..."
        }}
    ]
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return PaperVerification(**result)

In [33]:
paper_verifications = []

for index, evidence in enumerate(
    paper_evidence_store,
    start=1
):
    print(
        f"Verifying {index}/{len(paper_evidence_store)}: "
        f"{evidence.source_title}"
    )

    try:
        verification = verify_paper_evidence(
            evidence
        )

        if verification:
            paper_verifications.append(
                verification
            )

            print("✅ Verification completed.")
        else:
            print("⚠️ Verification skipped.")

    except Exception as e:
        print(
            f"❌ Verification failed: {e}"
        )

    print()

print(
    f"✅ Verified papers: "
    f"{len(paper_verifications)}"
)

Verifying 1/3: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Verification completed.

Verifying 2/3: Machine learning in solar physics
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Verification completed.

Verifying 3/3: The flare likelihood and region eruption forecasting (FLARECAST) project: flare forecasting in the big data & machine learning era
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Verification completed.

✅ Verified papers: 3


In [34]:
verification_summary = {
    "supported": 0,
    "partially_supported": 0,
    "unsupported": 0
}

for paper_verification in paper_verifications:
    for claim in paper_verification.verified_claims:

        if claim.status in verification_summary:
            verification_summary[
                claim.status
            ] += 1

print("Verification Summary")
print("--------------------")

for status, count in verification_summary.items():
    print(f"{status}: {count}")

Verification Summary
--------------------
supported: 21
partially_supported: 0
unsupported: 0


In [35]:
trusted_claims = []

for paper_verification in paper_verifications:
    for claim in paper_verification.verified_claims:

        if claim.status == "supported":
            trusted_claims.append({
                "paper_id": paper_verification.paper_id,
                "source_title": paper_verification.source_title,
                "claim": claim.claim,
                "status": claim.status,
                "explanation": claim.explanation
            })

print(
    f"✅ Trusted claims: "
    f"{len(trusted_claims)}"
)

✅ Trusted claims: 21


In [36]:
review_claims = []

for paper_verification in paper_verifications:
    for claim in paper_verification.verified_claims:

        if claim.status != "supported":
            review_claims.append({
                "paper_id": paper_verification.paper_id,
                "source_title": paper_verification.source_title,
                "claim": claim.claim,
                "status": claim.status,
                "explanation": claim.explanation
            })

print(
    f"Claims requiring review: "
    f"{len(review_claims)}"
)

Claims requiring review: 0


## Phase 8 — Cross-Source Evidence Relations

This stage compares verified scientific claims across different papers.

Responsibilities:

- identify supporting evidence across sources
- identify complementary findings
- detect tensions caused by different assumptions or experimental settings
- detect genuine contradictions
- avoid treating every difference as a contradiction

In [37]:
claim_records = []

for index, item in enumerate(
    trusted_claims,
    start=1
):
    claim_records.append({
        "claim_id": f"C{index}",
        "paper_id": item["paper_id"],
        "source_title": item["source_title"],
        "claim": item["claim"]
    })

print(
    f"Claims prepared for comparison: "
    f"{len(claim_records)}"
)

Claims prepared for comparison: 21


In [38]:
class ClaimRelation(BaseModel):
    claim_id_a: str
    claim_id_b: str
    topic: str
    relation: str
    explanation: str


class CrossSourceReport(BaseModel):
    relations: List[ClaimRelation] = Field(
        default_factory=list
    )
    summary: str

In [39]:
def detect_cross_source_relations(
    claim_records: list
):
    claims_text = json.dumps(
        claim_records,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Cross-Source Scientific Evidence Agent of AURA.

You are given VERIFIED claims extracted from scientific paper abstracts.

Your task is to compare claims that discuss the same or closely
related scientific topics.

CLAIMS:

{claims_text}

For meaningful claim pairs, assign exactly one relationship:

- supports
- complements
- tension
- contradicts

Definitions:

supports:
Two claims independently provide similar evidence or conclusions.

complements:
The claims provide different but compatible information about
the same topic.

tension:
The claims differ, but the difference may result from different
methods, datasets, assumptions, targets, or experimental settings.

contradicts:
The claims cannot reasonably both be true under the same scope
and conditions.

Important rules:

1. Do not use outside knowledge.
2. Do not compare unrelated claims.
3. Do not call something a contradiction merely because results differ.
4. Preserve differences in scope.
5. Do not invent experimental details.
6. Return at most the 12 most informative relationships.
7. Only compare claims from DIFFERENT papers.
   Never create a relationship between two claims from the same source_title
   or the same paper_id.

Return ONLY valid JSON:

{{
    "relations": [
        {{
            "claim_id_a": "C1",
            "claim_id_b": "C2",
            "topic": "...",
            "relation": "supports",
            "explanation": "..."
        }}
    ],
    "summary": "Concise description of the overall evidence relationships."
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=DEFAULT_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return CrossSourceReport(**result)

In [40]:
cross_source_report = detect_cross_source_relations(
    claim_records
)

print(
    json.dumps(
        cross_source_report.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash
✅ Success with: gemini-3.5-flash
{
  "relations": [
    {
      "claim_id_a": "C1",
      "claim_id_b": "C7",
      "topic": "Application of Machine Learning to Solar Flares",
      "relation": "supports",
      "explanation": "C1 states that machine learning has seen significant activity in forecasting solar flares, which is supported by C7's claim that machine learning improves the understanding of explosive events like solar flares by allowing deeper data analysis."
    },
    {
      "claim_id_a": "C1",
      "claim_id_b": "C14",
      "topic": "Machine Learning for Solar Flare Forecasting",
      "relation": "complements",
      "explanation": "C1 highlights that solar flare forecasting is a major area of ML activity, while C14 provides concrete details on how this is achieved in practice (using 170+ properties in multiple ML algorithms to forecast different flaring levels)."
    },
    {
      "claim_id_a": "C1",
      "claim_id_b": "C16",
      "to

In [41]:
print(
    f"Relations detected: "
    f"{len(cross_source_report.relations)}"
)

relation_counts = {
    "supports": 0,
    "complements": 0,
    "tension": 0,
    "contradicts": 0
}

for relation in cross_source_report.relations:
    if relation.relation in relation_counts:
        relation_counts[relation.relation] += 1

for relation_type, count in relation_counts.items():
    print(f"{relation_type}: {count}")

Relations detected: 10
supports: 4
complements: 6
tension: 0
contradicts: 0


## Phase 9 — Evidence Synthesis & Analysis

This stage synthesizes verified evidence across multiple scientific sources.

Responsibilities:

- identify evidence-backed findings
- identify recurring methodological patterns
- compare methods and evidence across papers
- preserve uncertainty
- identify unresolved evidence gaps
- avoid conclusions broader than the reviewed evidence

In [42]:
class AnalysisResult(BaseModel):
    key_findings: List[str] = Field(default_factory=list)
    methods_identified: List[str] = Field(default_factory=list)
    data_sources_identified: List[str] = Field(default_factory=list)
    methodological_patterns: List[str] = Field(default_factory=list)
    uncertainties: List[str] = Field(default_factory=list)
    evidence_gaps: List[str] = Field(default_factory=list)
    synthesis: str

In [43]:
def normalize_string_list(
    value,
    preferred_keys=None
):
    if preferred_keys is None:
        preferred_keys = [
            "finding",
            "method",
            "data_source",
            "pattern",
            "uncertainty",
            "gap",
            "text",
            "value"
        ]

    if not isinstance(value, list):
        return []

    normalized = []

    for item in value:

        if isinstance(item, str):
            normalized.append(item)
            continue

        if isinstance(item, dict):

            extracted = None

            for key in preferred_keys:
                candidate = item.get(key)

                if isinstance(candidate, str):
                    extracted = candidate
                    break

            if extracted:
                normalized.append(extracted)

    return normalized

In [44]:
def analyze_verified_evidence(
    trusted_claims: list,
    cross_source_report: CrossSourceReport
):
    claims_payload = [
        {
            "source_title": item["source_title"],
            "claim": item["claim"]
        }
        for item in trusted_claims
    ]

    relations_payload = [
        relation.model_dump()
        for relation in cross_source_report.relations
    ]

    claims_text = json.dumps(
        claims_payload,
        indent=2,
        ensure_ascii=False
    )

    relations_text = json.dumps(
        relations_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Evidence Synthesis and Analysis Agent of AURA.

Use ONLY the VERIFIED scientific claims and genuine cross-source
relationships supplied below.

VERIFIED CLAIMS:

{claims_text}

CROSS-SOURCE RELATIONSHIPS:

{relations_text}

Rules:

1. Do not use outside knowledge.
2. Do not invent methods, datasets, results, or limitations.
3. Do not generalize a result from one paper into a universal conclusion.
4. Preserve the scope of each source-specific finding.
5. Distinguish repeated evidence from synthesis.
6. Preserve uncertainty.
7. If evidence is insufficient, record an evidence gap.
8. Do not treat absence of evidence as evidence of absence.
9. Do not produce a final implementation recommendation yet.
10. Avoid words such as "best", "optimal", or "significantly improved"
    unless the verified evidence explicitly supports that wording and scope.
11. Every item inside key_findings, methods_identified,
    data_sources_identified, methodological_patterns,
    uncertainties, and evidence_gaps MUST be a plain string.
12. Do not return objects such as
    {{"finding": "..."}} or {{"method": "..."}} inside those arrays.

Return ONLY valid JSON:

{{
    "key_findings": [],
    "methods_identified": [],
    "data_sources_identified": [],
    "methodological_patterns": [],
    "uncertainties": [],
    "evidence_gaps": [],
    "synthesis": "..."
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    normalized_result = {
        "key_findings": normalize_string_list(
            result.get("key_findings", []),
            ["finding", "text", "value"]
        ),

        "methods_identified": normalize_string_list(
            result.get("methods_identified", []),
            ["method", "text", "value"]
        ),

        "data_sources_identified": normalize_string_list(
            result.get("data_sources_identified", []),
            ["data_source", "dataset", "source", "text", "value"]
        ),

        "methodological_patterns": normalize_string_list(
            result.get("methodological_patterns", []),
            ["pattern", "text", "value"]
        ),

        "uncertainties": normalize_string_list(
            result.get("uncertainties", []),
            ["uncertainty", "text", "value"]
        ),

        "evidence_gaps": normalize_string_list(
            result.get("evidence_gaps", []),
            ["gap", "text", "value"]
        ),

        "synthesis": result.get(
            "synthesis",
            ""
        )
    }

    return AnalysisResult(
        **normalized_result
    )

In [45]:
analysis_result = analyze_verified_evidence(
    trusted_claims,
    cross_source_report
)

print(
    json.dumps(
        analysis_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)


Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "key_findings": [
    "Machine learning has seen the most activity in forecasting geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flares occurrence, coronal mass ejection propagation time, and solar wind speed.",
    "Machine learning can analyze large amounts of data from solar observations to identify patterns and trends not apparent using traditional methods.",
    "Machine learning improves our understanding of explosive events like solar flares and the inner workings of the sun by allowing deeper data analysis and more complex models.",
    "Machine learning automates the analysis of solar data, reducing manual labor and increasing research efficiency.",
    "Combined use of 170+ properties in multiple machine-learning algorithms gave rise to changing sets of best-performing predictors for the forecasting of different flaring levels, at least for major flares.",
    "Rigorous 

## Phase 10 — Dataset Discovery

This stage searches for real datasets related to the research question.

Responsibilities:

- use the dataset-search task generated by the planner
- retrieve dataset metadata from DataCite
- relax overly specific queries when needed
- remove duplicate records
- preserve DOI, publisher, year, and provenance

In [46]:
class DatasetSource(BaseModel):
    title: str
    creators: List[str] = Field(default_factory=list)
    year: int | None = None
    publisher: str | None = None
    description: str | None = None
    doi: str | None = None
    url: str | None = None
    resource_type: str | None = None

In [47]:
dataset_tasks = [
    task
    for task in plan.tasks
    if task.task_type == "dataset_search"
]

dataset_query = dataset_tasks[0].query

print("Dataset search query:")
print(dataset_query)

Dataset search query:
SDO HMI magnetogram dataset solar flare prediction benchmark dataset SWAN-SF


In [48]:
def search_datacite_datasets(
    query: str,
    limit: int = 5
):
    url = "https://api.datacite.org/dois"

    params = {
        "query": query,
        "resource-type-id": "dataset",
        "page[size]": limit
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [49]:
dataset_queries = [
    dataset_query,
    "solar flare",
    "solar flare prediction",
    "solar activity",
    "SHARP HMI",
    "GOES solar flare"
]

raw_datasets = None
successful_dataset_query = None

for query in dataset_queries:
    print(f"Trying dataset query: {query}")

    result = search_datacite_datasets(
        query=query,
        limit=5
    )

    count = len(
        result.get("data", [])
    )

    print(f"Datasets returned: {count}")
    print()

    if count > 0:
        raw_datasets = result
        successful_dataset_query = query
        break


if raw_datasets is not None:
    print(
        f"✅ Dataset search succeeded with: "
        f"{successful_dataset_query}"
    )
else:
    print("⚠️ No datasets found.")

Trying dataset query: SDO HMI magnetogram dataset solar flare prediction benchmark dataset SWAN-SF
Datasets returned: 0

Trying dataset query: solar flare
Datasets returned: 5

✅ Dataset search succeeded with: solar flare


In [50]:
def normalize_datacite_datasets(
    search_result
):
    datasets = []

    for item in search_result.get("data", []):
        attributes = item.get(
            "attributes",
            {}
        )

        titles = attributes.get(
            "titles",
            []
        )

        title = (
            titles[0].get("title")
            if titles
            else ""
        )

        creators = []

        for creator in attributes.get(
            "creators",
            []
        ):
            name = creator.get("name")

            if name:
                creators.append(name)

        descriptions = attributes.get(
            "descriptions",
            []
        )

        description = None

        if descriptions:
            description = descriptions[0].get(
                "description"
            )

        doi = attributes.get("doi")

        dataset = DatasetSource(
            title=title,
            creators=creators,
            year=attributes.get(
                "publicationYear"
            ),
            publisher=attributes.get(
                "publisher"
            ),
            description=description,
            doi=doi,
            url=(
                f"https://doi.org/{doi}"
                if doi
                else attributes.get("url")
            ),
            resource_type=(
                attributes
                .get("types", {})
                .get("resourceTypeGeneral")
            )
        )

        datasets.append(dataset)

    return datasets

In [51]:
datasets = normalize_datacite_datasets(
    raw_datasets
)

print(
    f"✅ {len(datasets)} datasets normalized."
)

✅ 5 datasets normalized.


In [52]:
def deduplicate_datasets_by_title(
    datasets
):
    unique_datasets = []
    seen_titles = set()

    for dataset in datasets:
        normalized_title = (
            dataset.title
            .strip()
            .lower()
        )

        if normalized_title not in seen_titles:
            unique_datasets.append(
                dataset
            )

            seen_titles.add(
                normalized_title
            )

    return unique_datasets

In [53]:
unique_datasets = (
    deduplicate_datasets_by_title(
        datasets
    )
)

print(f"Raw datasets: {len(datasets)}")
print(
    f"Unique datasets: "
    f"{len(unique_datasets)}"
)

Raw datasets: 5
Unique datasets: 3


In [54]:
for index, dataset in enumerate(
    unique_datasets,
    start=1
):
    print("=" * 90)
    print(f"DATASET #{index}")
    print(f"Title: {dataset.title}")
    print(f"Year: {dataset.year}")
    print(
        f"Publisher: "
        f"{dataset.publisher}"
    )
    print(f"DOI: {dataset.doi}")
    print(f"URL: {dataset.url}")

    if dataset.description:
        print(
            f"Description: "
            f"{dataset.description[:700]}"
        )

    print()

DATASET #1
Title: DST/SSODA Ca II K Flare Catalogue v1.1
Year: 2026
Publisher: Zenodo
DOI: 10.5281/zenodo.20710751
URL: https://doi.org/10.5281/zenodo.20710751
Description: The DST/SSODA Ca II K flare catalogue supports the manuscript "A First Calibrated Catalogue of Ca II K Flare Signatures from the Dunn Solar Telescope." It contains 24 retained Ca II K detections: 18 selected by the GOES-guided search and 6 selected by the Ca II K search after masking the dominant GOES peak window. Detection mode describes the selection path; a GOES-independent detection may later be associated with a local GOES feature.

The ZIP contains the 24-row catalogue, a configured-sequence coverage table, SnK class definitions, 24 Ca II K light curves and compact enhancement masks, 21 usable event-window H-alpha light curves, 24 one-minute GOES/XRS-B context curves, per-event region 

DATASET #2
Title: Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integ

### Dataset Relevance Evaluation

Retrieved dataset records are evaluated against the original research goal.

This stage:

- identifies directly relevant datasets
- separates core resources from supporting resources
- preserves dataset provenance
- prevents weakly related datasets from influencing downstream decisions

In [55]:
class DatasetRelevanceAssessment(BaseModel):
    title: str
    doi: str | None = None
    relevance: str
    role: str
    keep: bool
    reason: str

In [56]:
def evaluate_dataset_relevance(
    user_question: str,
    datasets: list
):
    dataset_payload = []

    for dataset in datasets:
        dataset_payload.append({
            "title": dataset.title,
            "year": dataset.year,
            "publisher": dataset.publisher,
            "description": dataset.description,
            "doi": dataset.doi,
            "resource_type": dataset.resource_type
        })

    dataset_text = json.dumps(
        dataset_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Dataset Relevance Evaluator of AURA.

USER RESEARCH QUESTION:

{user_question}

CANDIDATE DATASETS:

{dataset_text}

Evaluate each dataset for usefulness to the user's
solar flare prediction research and implementation goal.

Rules:

1. relevance must be high, medium, or low.
2. role must be core or supporting.
3. keep=true only if the dataset could reasonably contribute
   to solar flare prediction research or implementation.
4. A core dataset should directly support model development,
   training, validation, or benchmarking for solar flare prediction.
5. A supporting dataset may provide useful auxiliary observations
   but should not be treated as the primary modeling dataset.
6. Use only the supplied metadata.
7. Do not infer undocumented capabilities.
8. If multiple records appear to be variants of the same dataset family,
   mention that in the reason.

Return ONLY valid JSON:

[
    {{
        "title": "...",
        "doi": "...",
        "relevance": "high",
        "role": "core",
        "keep": true,
        "reason": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return [
        DatasetRelevanceAssessment(**item)
        for item in result
    ]

In [57]:
dataset_relevance_results = evaluate_dataset_relevance(
    question,
    unique_datasets
)

for index, result in enumerate(
    dataset_relevance_results,
    start=1
):
    print("=" * 90)
    print(f"DATASET #{index}")
    print(f"Title: {result.title}")
    print(f"Relevance: {result.relevance}")
    print(f"Role: {result.role}")
    print(f"Keep: {result.keep}")
    print(f"Reason: {result.reason}")
    print()

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
DATASET #1
Title: DST/SSODA Ca II K Flare Catalogue v1.1
Relevance: medium
Role: supporting
Keep: True
Reason: Provides a localized flare signature catalogue with light curves and enhancement masks from the Dunn Solar Telescope, useful as auxiliary or validation data for flare signatures, though not structured as a primary machine learning training dataset for full-disk solar flare forecasting.

DATASET #2
Title: Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients
Relevance: high
Role: core
Keep: True
Reason: Directly provides cross-validation datasets, comparative datasets, and experimental code utilizing modern machine learning methods (Mamba) specifically designed for solar flare forecasting.

DATASET #3
Title: SEP-PRISM Data:   A multi-source dataset for solar energetic particle forecasting
Relevance: medium
Role: supporting
Keep: True
Rea

In [58]:
core_dataset_dois = {
    result.doi
    for result in dataset_relevance_results
    if result.keep and result.role == "core"
}

supporting_dataset_dois = {
    result.doi
    for result in dataset_relevance_results
    if result.keep and result.role == "supporting"
}

core_datasets = [
    dataset
    for dataset in unique_datasets
    if dataset.doi in core_dataset_dois
]

supporting_datasets = [
    dataset
    for dataset in unique_datasets
    if dataset.doi in supporting_dataset_dois
]

print(f"Core dataset records: {len(core_datasets)}")
print(f"Supporting dataset records: {len(supporting_datasets)}")

Core dataset records: 1
Supporting dataset records: 2


In [59]:
for dataset in core_datasets:
    print(dataset.title)

Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients


### Dataset Family Consolidation

Multiple dataset records may represent different variants of the same
underlying dataset family.

This stage groups related dataset variants while preserving their
individual DOI records and roles.

In [60]:
dataset_families = {}

for dataset in unique_datasets:

    if dataset.title.startswith(
        "Active region magnetograms for solar flare prediction"
    ):
        family_name = (
            "Active Region Magnetograms for Solar Flare Prediction"
        )

    elif dataset.title.startswith("JW-FD"):
        family_name = "JW-FD"

    else:
        family_name = dataset.title

    if family_name not in dataset_families:
        dataset_families[family_name] = []

    dataset_families[family_name].append(dataset)

In [61]:
print(
    f"Logical dataset families: "
    f"{len(dataset_families)}"
)

for family_name, members in dataset_families.items():

    print("=" * 90)
    print(f"FAMILY: {family_name}")
    print(f"Records: {len(members)}")

    for dataset in members:
        print(
            f"- {dataset.title} "
            f"({dataset.doi})"
        )

    print()

Logical dataset families: 3
FAMILY: DST/SSODA Ca II K Flare Catalogue v1.1
Records: 1
- DST/SSODA Ca II K Flare Catalogue v1.1 (10.5281/zenodo.20710751)

FAMILY: Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients
Records: 1
- Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients (10.5281/zenodo.22748717)

FAMILY: SEP-PRISM Data:   A multi-source dataset for solar energetic particle forecasting
Records: 1
- SEP-PRISM Data:   A multi-source dataset for solar energetic particle forecasting (10.5281/zenodo.21297634)



## Phase 11 — Code Discovery

This stage searches for real software repositories related to
solar flare prediction.

Responsibilities:

- use the code-search task generated by the planner
- search real GitHub repositories
- relax overly specific queries when necessary
- preserve repository metadata and provenance
- prepare repositories for relevance evaluation and README inspection

In [62]:
class CodeSource(BaseModel):
    name: str
    full_name: str
    description: str | None = None
    url: str
    language: str | None = None
    stars: int = 0
    forks: int = 0
    updated_at: str | None = None

In [63]:
code_tasks = [
    task
    for task in plan.tasks
    if task.task_type == "code_search"
]

code_query = code_tasks[0].query

print("Code search query:")
print(code_query)

Code search query:
solar flare prediction GitHub machine learning deep learning implementation


In [64]:
def search_github_repositories(
    query: str,
    limit: int = 5
):
    url = "https://api.github.com/search/repositories"

    params = {
        "q": query,
        "per_page": limit,
        "sort": "stars",
        "order": "desc"
    }

    headers = {
        "Accept": "application/vnd.github+json",
        "User-Agent": "AURA-Research-Agent"
    }

    github_token = os.getenv(
        "GITHUB_TOKEN"
    )

    if github_token:
        headers["Authorization"] = (
            f"Bearer {github_token}"
        )

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    if response.status_code in [403, 429]:
        return {
            "items": [],
            "_rate_limited": True,
            "_status_code": response.status_code
        }

    response.raise_for_status()

    result = response.json()

    result["_rate_limited"] = False

    return result

In [65]:
code_queries = [
    code_query,
    "solar flare forecasting",
    "solar flare prediction",
    "solar flare machine learning",
    "solar flare LSTM"
]

raw_code_results = None
successful_code_query = None
github_rate_limited = False

for query in code_queries:

    print(
        f"Trying code query: {query}"
    )

    result = search_github_repositories(
        query=query,
        limit=5
    )

    if result.get(
        "_rate_limited",
        False
    ):
        print(
            "⚠️ GitHub API rate limit reached."
        )

        github_rate_limited = True
        break

    count = len(
        result.get("items", [])
    )

    print(
        f"Repositories returned: {count}"
    )
    print()

    if count > 0:
        raw_code_results = result
        successful_code_query = query
        break

Trying code query: solar flare prediction GitHub machine learning deep learning implementation
⚠️ GitHub API rate limit reached.


In [66]:
if (
    raw_code_results is None
    and github_rate_limited
):
    print(
        "⚠️ Using cached repository metadata "
        "from the previous successful AURA run."
    )

    raw_code_results = {
        "items": [
            {
                "name": "flare_duration_forecasting",
                "full_name": (
                    "USNavalResearchLaboratory/"
                    "flare_duration_forecasting"
                ),
                "description": (
                    "Solar Flare Duration Forecasting"
                ),
                "html_url": (
                    "https://github.com/"
                    "USNavalResearchLaboratory/"
                    "flare_duration_forecasting"
                ),
                "language": "Jupyter Notebook",
                "stargazers_count": 8,
                "forks_count": 4,
                "updated_at": (
                    "2025-10-07T21:51:45Z"
                )
            },
            {
                "name": "solar_flares_forecasting",
                "full_name": (
                    "iknyazeva/"
                    "solar_flares_forecasting"
                ),
                "description": (
                    "Solar flare forecasting with "
                    "time-series feature engineering."
                ),
                "html_url": (
                    "https://github.com/"
                    "iknyazeva/"
                    "solar_flares_forecasting"
                ),
                "language": "Jupyter Notebook",
                "stargazers_count": 6,
                "forks_count": 0,
                "updated_at": (
                    "2026-06-29T19:09:40Z"
                )
            },
            {
                "name": "solar-flare-forecaster",
                "full_name": (
                    "inaf-oact-ai/"
                    "solar-flare-forecaster"
                ),
                "description": (
                    "Solar flare forecaster application "
                    "based on transformer models"
                ),
                "html_url": (
                    "https://github.com/"
                    "inaf-oact-ai/"
                    "solar-flare-forecaster"
                ),
                "language": "Python",
                "stargazers_count": 3,
                "forks_count": 0,
                "updated_at": (
                    "2026-07-27T14:33:30Z"
                )
            },
            {
                "name": "flare-forecasting",
                "full_name": (
                    "ulrichw/"
                    "flare-forecasting"
                ),
                "description": (
                    "Python and R machine-learning codes "
                    "for the forecasting of solar flares"
                ),
                "html_url": (
                    "https://github.com/"
                    "ulrichw/"
                    "flare-forecasting"
                ),
                "language": "Python",
                "stargazers_count": 2,
                "forks_count": 0,
                "updated_at": (
                    "2022-11-08T02:31:23Z"
                )
            },
            {
                "name": "Sola",
                "full_name": "domijan/Sola",
                "description": (
                    'Code for "Solar Flare Forecasting '
                    'from Magnetic Feature Properties '
                    'Generated by SMART"'
                ),
                "html_url": (
                    "https://github.com/"
                    "domijan/Sola"
                ),
                "language": "R",
                "stargazers_count": 2,
                "forks_count": 1,
                "updated_at": (
                    "2024-03-02T04:22:53Z"
                )
            }
        ]
    }

    successful_code_query = (
        "cached previous successful GitHub search"
    )


if raw_code_results is not None:
    print(
        f"✅ Code source available: "
        f"{successful_code_query}"
    )
else:
    print(
        "⚠️ No repositories available."
    )

⚠️ Using cached repository metadata from the previous successful AURA run.
✅ Code source available: cached previous successful GitHub search


In [67]:
def normalize_github_repositories(
    search_result
):
    repositories = []

    for item in search_result.get("items", []):

        repository = CodeSource(
            name=item.get("name", ""),
            full_name=item.get("full_name", ""),
            description=item.get("description"),
            url=item.get("html_url", ""),
            language=item.get("language"),
            stars=item.get(
                "stargazers_count",
                0
            ),
            forks=item.get(
                "forks_count",
                0
            ),
            updated_at=item.get(
                "updated_at"
            )
        )

        repositories.append(
            repository
        )

    return repositories

In [68]:
code_sources = normalize_github_repositories(
    raw_code_results
)

print(
    f"✅ {len(code_sources)} repositories normalized."
)

✅ 5 repositories normalized.


In [69]:
for index, repo in enumerate(
    code_sources,
    start=1
):
    print("=" * 90)
    print(f"REPOSITORY #{index}")
    print(f"Name: {repo.full_name}")
    print(f"Description: {repo.description}")
    print(f"Language: {repo.language}")
    print(f"Stars: {repo.stars}")
    print(f"Forks: {repo.forks}")
    print(f"Updated: {repo.updated_at}")
    print(f"URL: {repo.url}")
    print()

REPOSITORY #1
Name: USNavalResearchLaboratory/flare_duration_forecasting
Description: Solar Flare Duration Forecasting
Language: Jupyter Notebook
Stars: 8
Forks: 4
Updated: 2025-10-07T21:51:45Z
URL: https://github.com/USNavalResearchLaboratory/flare_duration_forecasting

REPOSITORY #2
Name: iknyazeva/solar_flares_forecasting
Description: Solar flare forecasting with time-series feature engineering.
Language: Jupyter Notebook
Stars: 6
Forks: 0
Updated: 2026-06-29T19:09:40Z
URL: https://github.com/iknyazeva/solar_flares_forecasting

REPOSITORY #3
Name: inaf-oact-ai/solar-flare-forecaster
Description: Solar flare forecaster application based on transformer models
Language: Python
Stars: 3
Forks: 0
Updated: 2026-07-27T14:33:30Z
URL: https://github.com/inaf-oact-ai/solar-flare-forecaster

REPOSITORY #4
Name: ulrichw/flare-forecasting
Description: Python and R machine-learning codes for the forecasting of solar flares
Language: Python
Stars: 2
Forks: 0
Updated: 2022-11-08T02:31:23Z
URL: http

### Code Relevance Evaluation

Retrieved repositories are evaluated against the original research goal
before repository documentation is inspected.

This stage:

- identifies directly relevant solar flare forecasting implementations
- separates core repositories from supporting implementations
- preserves repository provenance
- reduces unnecessary README retrieval and downstream processing

In [70]:
class CodeRelevanceAssessment(BaseModel):
    full_name: str
    url: str
    relevance: str
    role: str
    keep: bool
    reason: str

In [71]:
def evaluate_code_relevance(
    user_question: str,
    repositories: list
):
    repo_payload = []

    for repo in repositories:
        repo_payload.append({
            "full_name": repo.full_name,
            "description": repo.description,
            "language": repo.language,
            "stars": repo.stars,
            "forks": repo.forks,
            "updated_at": repo.updated_at,
            "url": repo.url
        })

    repo_text = json.dumps(
        repo_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Code Relevance Evaluator of AURA.

USER RESEARCH QUESTION:

{user_question}

CANDIDATE CODE REPOSITORIES:

{repo_text}

Evaluate each repository for usefulness to the user's
solar flare prediction research and implementation goal.

Rules:

1. relevance must be high, medium, or low.
2. role must be core or supporting.
3. keep=true if the repository could reasonably contribute
   to implementation, comparison, benchmarking, or reproducibility.
4. Core repositories should directly implement solar flare
   forecasting or prediction.
5. Supporting repositories may address related tasks such as
   flare duration forecasting or auxiliary feature analysis.
6. Do not assume code quality from star count.
7. Use only the supplied repository metadata.
8. If metadata is insufficient, say so explicitly.

Return ONLY valid JSON:

[
    {{
        "full_name": "...",
        "url": "...",
        "relevance": "high",
        "role": "core",
        "keep": true,
        "reason": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return [
        CodeRelevanceAssessment(**item)
        for item in result
    ]

In [72]:
code_relevance_results = evaluate_code_relevance(
    question,
    code_sources
)

for index, result in enumerate(
    code_relevance_results,
    start=1
):
    print("=" * 90)
    print(f"REPOSITORY #{index}")
    print(f"Name: {result.full_name}")
    print(f"Relevance: {result.relevance}")
    print(f"Role: {result.role}")
    print(f"Keep: {result.keep}")
    print(f"Reason: {result.reason}")
    print()

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
REPOSITORY #1
Name: inaf-oact-ai/solar-flare-forecaster
Relevance: high
Role: core
Keep: True
Reason: Directly implements solar flare forecasting using modern transformer models, making it highly valuable for the core machine learning implementation.

REPOSITORY #2
Name: iknyazeva/solar_flares_forecasting
Relevance: high
Role: core
Keep: True
Reason: Focuses specifically on solar flare forecasting with time-series feature engineering, providing direct utility for predictive modeling and feature extraction approaches.

REPOSITORY #3
Name: ulrichw/flare-forecasting
Relevance: high
Role: core
Keep: True
Reason: Provides Python and R machine-learning codes explicitly designed for forecasting solar flares, offering alternative methodological baselines.

REPOSITORY #4
Name: domijan/Sola
Relevance: high
Role: core
Keep: True
Reason: Implements solar flare forecasting based on magnetic feature properties generated by SMA

In [73]:
core_code_urls = {
    result.url
    for result in code_relevance_results
    if result.keep
    and result.role == "core"
}

supporting_code_urls = {
    result.url
    for result in code_relevance_results
    if result.keep
    and result.role == "supporting"
}

core_code_sources = [
    repo
    for repo in code_sources
    if repo.url in core_code_urls
]

supporting_code_sources = [
    repo
    for repo in code_sources
    if repo.url in supporting_code_urls
]

print(
    f"Core code sources: "
    f"{len(core_code_sources)}"
)

print(
    f"Supporting code sources: "
    f"{len(supporting_code_sources)}"
)

Core code sources: 4
Supporting code sources: 1


## Phase 12 — Repository Inspection & Code Evidence Extraction

This stage inspects repository documentation before using code
implementations as technical evidence.

Responsibilities:

- retrieve repository README files
- verify implementation details against repository documentation
- identify methods, data sources, frameworks, and available artifacts
- preserve implementation limitations
- avoid assumptions based only on repository names or descriptions

In [74]:
def fetch_github_readme(
    full_name: str
):
    branches = [
        "main",
        "master"
    ]

    filenames = [
        "README.md",
        "README.rst",
        "README"
    ]

    headers = {
        "User-Agent": "AURA-Research-Agent"
    }

    for branch in branches:

        for filename in filenames:

            url = (
                "https://raw.githubusercontent.com/"
                f"{full_name}/"
                f"{branch}/"
                f"{filename}"
            )

            try:
                response = requests.get(
                    url,
                    headers=headers,
                    timeout=30
                )

                if response.status_code == 200:
                    return response.text

            except requests.RequestException:
                continue

    return None

In [75]:
selected_code_sources = (
    core_code_sources
    + supporting_code_sources
)

print(
    f"Repositories selected for inspection: "
    f"{len(selected_code_sources)}"
)

Repositories selected for inspection: 5


In [76]:
repo_readme_store = {}

for repo in selected_code_sources:

    print(
        f"Fetching README: "
        f"{repo.full_name}"
    )

    try:
        readme = fetch_github_readme(
            repo.full_name
        )

        repo_readme_store[
            repo.full_name
        ] = readme

        if readme:
            print(
                f"✅ README retrieved "
                f"({len(readme)} characters)"
            )
        else:
            print(
                "⚠️ README not available."
            )

    except Exception as e:
        repo_readme_store[
            repo.full_name
        ] = None

        print(
            f"❌ README retrieval failed: {e}"
        )

    print()

Fetching README: iknyazeva/solar_flares_forecasting
✅ README retrieved (972 characters)

Fetching README: inaf-oact-ai/solar-flare-forecaster
✅ README retrieved (14388 characters)

Fetching README: ulrichw/flare-forecasting
✅ README retrieved (294 characters)

Fetching README: domijan/Sola
✅ README retrieved (88 characters)

Fetching README: USNavalResearchLaboratory/flare_duration_forecasting
✅ README retrieved (425 characters)



In [77]:
for repo_name, readme in (
    repo_readme_store.items()
):
    print("=" * 90)
    print(repo_name)
    print()

    if readme:
        print(readme[:1200])
    else:
        print("README unavailable.")

    print()

iknyazeva/solar_flares_forecasting

# solar_flares_forecasting
The sun produces solar flares, which have the power to affect the Earth and near-Earth environment with their great bursts of electromagnetic energy and particles. These flares have the power to blow out transformers on power grids and disrupt satellite systems. There is a long lasting task of predictions such events for minimizing its negative impact. Doing so is a difficult task because of the rarity of these events. The success in this task not changes significantly over the last 60 years. Actually, this was a topic of my Ph.D. research, and I did it without any machine learning. But either in the era of big data, there is no big success in this task. The most common approach described in the paper Bobra et al., 2014. The main drawback of the approach is ignoring time dependence on features. Here I tried to use knowledge about working with time series in features. All data for this project could be downloaded from the da

In [78]:
class CodeImplementationEvidence(
    BaseModel
):
    full_name: str
    repository_url: str
    role: str

    implementation_goal: str

    methods: List[str] = Field(
        default_factory=list
    )

    datasets_or_data_sources: List[str] = Field(
        default_factory=list
    )

    frameworks_or_languages: List[str] = Field(
        default_factory=list
    )

    available_artifacts: List[str] = Field(
        default_factory=list
    )

    limitations_or_missing_information: List[str] = Field(
        default_factory=list
    )

    evidence_summary: str

In [79]:
def extract_code_evidence(
    repositories: list,
    readme_store: dict,
    relevance_results: list
):
    repo_payload = []

    for repo in repositories:

        readme = readme_store.get(
            repo.full_name
        )

        if not readme:
            continue

        assessment = next(
            (
                result
                for result
                in relevance_results
                if result.url == repo.url
            ),
            None
        )

        role = (
            assessment.role
            if assessment
            else "supporting"
        )

        repo_payload.append({
            "full_name": repo.full_name,
            "repository_url": repo.url,
            "role": role,
            "repository_description": (
                repo.description
            ),
            "readme": readme[:15000]
        })

    repo_text = json.dumps(
        repo_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Code Implementation Evidence Agent of AURA.

Extract structured implementation evidence ONLY from the supplied
GitHub repository metadata and README content.

REPOSITORIES:

{repo_text}

Rules:

1. Do not use outside knowledge.
2. Do not infer methods that are not explicitly documented.
3. Do not infer datasets only from repository names.
4. Preserve documented framework, language, model, and dataset names.
5. Record missing implementation details as limitations.
6. Record obsolete dependencies or runtime requirements when explicitly documented.
7. Do not rank repositories.
8. Preserve the supplied repository role.
9. Every array item MUST be a plain string.

Return ONLY valid JSON as an array:

[
    {{
        "full_name": "...",
        "repository_url": "...",
        "role": "core",
        "implementation_goal": "...",
        "methods": [],
        "datasets_or_data_sources": [],
        "frameworks_or_languages": [],
        "available_artifacts": [],
        "limitations_or_missing_information": [],
        "evidence_summary": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return [
        CodeImplementationEvidence(
            **item
        )
        for item in result
    ]

In [80]:
code_evidence_store = extract_code_evidence(
    selected_code_sources,
    repo_readme_store,
    code_relevance_results
)

print(
    f"✅ Code evidence items: "
    f"{len(code_evidence_store)}"
)

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Code evidence items: 5


In [81]:
for index, evidence in enumerate(
    code_evidence_store,
    start=1
):
    print("=" * 90)
    print(
        f"CODE EVIDENCE #{index}"
    )
    print(
        f"Repository: "
        f"{evidence.full_name}"
    )
    print(
        f"Role: {evidence.role}"
    )
    print(
        f"Goal: "
        f"{evidence.implementation_goal}"
    )
    print(
        f"Methods: {evidence.methods}"
    )
    print(
        f"Data sources: "
        f"{evidence.datasets_or_data_sources}"
    )
    print(
        f"Frameworks/Languages: "
        f"{evidence.frameworks_or_languages}"
    )
    print(
        f"Artifacts: "
        f"{evidence.available_artifacts}"
    )
    print(
        f"Limitations: "
        f"{evidence.limitations_or_missing_information}"
    )
    print()

CODE EVIDENCE #1
Repository: iknyazeva/solar_flares_forecasting
Role: core
Goal: Solar flare forecasting with time-series feature engineering, taking into account time dependence on features.
Methods: ['time-series feature engineering']
Data sources: ['Data downloadable from the provided data link']
Frameworks/Languages: []
Artifacts: []
Limitations: ['Missing detailed framework or language specifications', 'Missing specific model architecture details beyond time-series feature engineering']

CODE EVIDENCE #2
Repository: inaf-oact-ai/solar-flare-forecaster
Role: core
Goal: Implement a solar flare forecasting application with modern transformer backbones across images, videos, and time series using a consistent training and evaluation framework.
Methods: ['SigLIP2 image forecaster with a ViT encoder and lightweight classifier', 'VideoMAE video forecaster with a spatio-temporal ViT backbone and classification head', 'Moirai2 time-series forecaster with a transformer encoder', 'Class-reba

## Phase 13 — Technical Documentation Discovery

This stage retrieves trusted technical documentation relevant to
solar flare research and implementation.

Responsibilities:

- retrieve official technical documentation
- preserve provider and source provenance
- identify data products and access methods
- extract implementation-relevant technical information
- avoid relying on unofficial documentation when authoritative sources exist

In [82]:
class DocumentationSource(BaseModel):
    name: str
    provider: str
    url: str
    topic: str
    content: str | None = None
    retrieval_status: str

In [83]:
documentation_registry = [
    {
        "name": "SunPy Documentation",
        "provider": "SunPy Project",
        "url": "https://docs.sunpy.org/en/stable/",
        "topic": (
            "Solar data access, processing, Python tools, "
            "and remote data interfaces"
        )
    },
    {
        "name": "SDO Data Access",
        "provider": "NASA Solar Dynamics Observatory",
        "url": "https://sdo.gsfc.nasa.gov/data/dataaccess.php",
        "topic": (
            "SDO/HMI data access, observations, "
            "and science data archives"
        )
    },
    {
        "name": "GOES X-ray Flux",
        "provider": "NOAA Space Weather Prediction Center",
        "url": "https://www.swpc.noaa.gov/products/goes-x-ray-flux",
        "topic": (
            "GOES X-ray measurements, solar flare monitoring, "
            "and data access"
        )
    }
]

print(
    f"Documentation sources registered: "
    f"{len(documentation_registry)}"
)

Documentation sources registered: 3


In [84]:
def fetch_documentation_page(
    url: str
):
    headers = {
        "User-Agent": "AURA Research Agent/0.1"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    for element in soup([
        "script",
        "style",
        "nav",
        "footer"
    ]):
        element.decompose()

    text = soup.get_text(
        separator=" ",
        strip=True
    )

    return text

In [85]:
documentation_sources = []

for source in documentation_registry:

    print(
        f"Fetching: {source['name']}"
    )

    try:
        content = fetch_documentation_page(
            source["url"]
        )

        documentation_sources.append(
            DocumentationSource(
                name=source["name"],
                provider=source["provider"],
                url=source["url"],
                topic=source["topic"],
                content=content,
                retrieval_status="success"
            )
        )

        print(
            f"✅ Retrieved "
            f"({len(content)} characters)"
        )

    except Exception as e:
        documentation_sources.append(
            DocumentationSource(
                name=source["name"],
                provider=source["provider"],
                url=source["url"],
                topic=source["topic"],
                content=None,
                retrieval_status="failed"
            )
        )

        print(
            f"❌ Failed: {e}"
        )

    print()

Fetching: SunPy Documentation
✅ Retrieved (1258 characters)

Fetching: SDO Data Access
✅ Retrieved (4644 characters)

Fetching: GOES X-ray Flux
✅ Retrieved (5506 characters)



In [86]:
for source in documentation_sources:

    print("=" * 90)
    print(
        f"DOCUMENTATION: "
        f"{source.name}"
    )
    print(
        f"Provider: "
        f"{source.provider}"
    )
    print(
        f"Status: "
        f"{source.retrieval_status}"
    )

    if source.content:
        print(
            f"Preview: "
            f"{source.content[:700]}"
        )
    else:
        print(
            "Preview: unavailable"
        )

    print()

DOCUMENTATION: SunPy Documentation
Provider: SunPy Project
Status: success
Preview: sunpy core Documentation — sunpy 8.0.0 documentation Skip to main content Back to top Ctrl + K SunPy Search Ctrl + K System Settings Light Dark Search Ctrl + K System Settings Light Dark sunpy 8.0.0 documentation sunpy core Documentation # sunpy is a community-developed, free and open-source solar data analysis environment for Python.
It includes an interface for searching and downloading data from multiple data providers, data containers for image and time series data, commonly used solar coordinate frames and associated transformations, as well as other functionality needed for solar data analysis. Tutorial New users start here! Walkthrough on how to install sunpy and use the key features 

DOCUMENTATION: SDO Data Access
Provider: NASA Solar Dynamics Observatory
Status: success
Preview: SDO | Solar Dynamics Observatory SDO | Data The Sun Now AIA/HMI Browse Data EVE L2 Browse Data Daily Movies Browse D

In [87]:
class DocumentationEvidence(BaseModel):
    source_name: str
    provider: str
    source_url: str

    capabilities: List[str] = Field(
        default_factory=list
    )

    data_products: List[str] = Field(
        default_factory=list
    )

    access_methods: List[str] = Field(
        default_factory=list
    )

    implementation_relevance: List[str] = Field(
        default_factory=list
    )

    limitations_or_missing_information: List[str] = Field(
        default_factory=list
    )

    evidence_summary: str

In [88]:
def extract_documentation_evidence(
    documentation_sources: list
):
    docs_payload = []

    for source in documentation_sources:

        if (
            source.retrieval_status != "success"
            or not source.content
        ):
            continue

        docs_payload.append({
            "source_name": source.name,
            "provider": source.provider,
            "source_url": source.url,
            "topic": source.topic,
            "content": source.content[:12000]
        })

    docs_text = json.dumps(
        docs_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Technical Documentation Evidence Agent of AURA.

Extract implementation-relevant technical evidence ONLY from
the supplied official documentation.

DOCUMENTATION:

{docs_text}

Rules:

1. Do not use outside knowledge.
2. Do not invent APIs, datasets, tools, or capabilities.
3. Preserve documented names of tools and data products.
4. Record missing implementation details as limitations.
5. Focus on information useful for solar flare research,
   data access, preprocessing, and implementation.
6. Every item inside arrays MUST be a plain string.

Return ONLY valid JSON as an array:

[
    {{
        "source_name": "...",
        "provider": "...",
        "source_url": "...",
        "capabilities": [],
        "data_products": [],
        "access_methods": [],
        "implementation_relevance": [],
        "limitations_or_missing_information": [],
        "evidence_summary": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return [
        DocumentationEvidence(**item)
        for item in result
    ]

In [89]:
documentation_evidence_store = (
    extract_documentation_evidence(
        documentation_sources
    )
)

print(
    f"✅ Documentation evidence items: "
    f"{len(documentation_evidence_store)}"
)

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Documentation evidence items: 3


In [90]:
for index, evidence in enumerate(
    documentation_evidence_store,
    start=1
):
    print("=" * 90)
    print(
        f"DOCUMENTATION EVIDENCE #{index}"
    )
    print(
        f"Source: {evidence.source_name}"
    )
    print(
        f"Provider: {evidence.provider}"
    )
    print(
        f"Capabilities: "
        f"{evidence.capabilities}"
    )
    print(
        f"Data products: "
        f"{evidence.data_products}"
    )
    print(
        f"Access methods: "
        f"{evidence.access_methods}"
    )
    print(
        f"Implementation relevance: "
        f"{evidence.implementation_relevance}"
    )
    print(
        f"Limitations: "
        f"{evidence.limitations_or_missing_information}"
    )
    print()

DOCUMENTATION EVIDENCE #1
Source: SunPy Documentation
Provider: SunPy Project
Capabilities: ['Searching and downloading data from multiple data providers', 'Data containers for image and time series data', 'Providing commonly used solar coordinate frames and associated transformations']
Data products: ['Image data containers', 'Time series data containers']
Access methods: ['Python interface for searching and downloading data']
Implementation relevance: ['Community-developed, free and open-source solar data analysis environment for Python', 'Provides foundational functionality needed for solar data analysis, coordinate systems, and remote data interfaces']
Limitations: ['Specific API methods, function names, and implementation syntax are not detailed in the provided text']

DOCUMENTATION EVIDENCE #2
Source: SDO Data Access
Provider: NASA Solar Dynamics Observatory
Capabilities: ['Browsing, downloading, and streaming solar images and animations', 'Creating time series plots of EVE Level

## Phase 14 — Unified Multi-Source Evidence Store

This stage combines evidence from scientific papers, datasets,
software repositories, and official technical documentation.

The unified store preserves:

- source type
- source provenance
- evidence status
- source role
- source URL
- implementation-relevant details

Evidence from different source types is not treated as having the same
verification status.

In [91]:
class UnifiedEvidenceItem(BaseModel):
    evidence_id: str

    source_type: str
    source_name: str
    source_url: str | None = None

    role: str
    evidence_status: str

    content: str

    details: List[str] = Field(
        default_factory=list
    )

    provenance_note: str

In [92]:
unified_evidence_store = []

evidence_counter = 1

for item in trusted_claims:

    paper = paper_lookup.get(
        item["paper_id"]
    )

    source_url = (
        paper.url
        if paper
        else None
    )

    unified_evidence_store.append(
        UnifiedEvidenceItem(
            evidence_id=f"E{evidence_counter}",
            source_type="paper",
            source_name=item["source_title"],
            source_url=source_url,
            role="scientific_evidence",
            evidence_status="verified_supported",
            content=item["claim"],
            details=[
                item["explanation"]
            ],
            provenance_note=(
                "Claim extracted from a scientific paper abstract "
                "and verified against the original abstract."
            )
        )
    )

    evidence_counter += 1

In [93]:
dataset_assessment_by_doi = {
    result.doi: result
    for result in dataset_relevance_results
}

In [94]:
for family_name, members in dataset_families.items():

    kept_members = []

    for dataset in members:

        assessment = dataset_assessment_by_doi.get(
            dataset.doi
        )

        if assessment and assessment.keep:
            kept_members.append(
                (
                    dataset,
                    assessment
                )
            )

    if not kept_members:
        continue

    roles = {
        assessment.role
        for _, assessment in kept_members
    }

    if roles == {"core"}:
        family_role = "core"

    elif roles == {"supporting"}:
        family_role = "supporting"

    else:
        family_role = "core+supporting"

    details = []

    for dataset, assessment in kept_members:

        detail = (
            f"Variant: {dataset.title} | "
            f"DOI: {dataset.doi} | "
            f"Year: {dataset.year} | "
            f"Publisher: {dataset.publisher} | "
            f"Role: {assessment.role}"
        )

        if dataset.description:
            detail += (
                f" | Description: "
                f"{dataset.description[:1200]}"
            )

        details.append(detail)

    source_url = (
        kept_members[0][0].url
        if len(kept_members) == 1
        else None
    )

    unified_evidence_store.append(
        UnifiedEvidenceItem(
            evidence_id=f"E{evidence_counter}",
            source_type="dataset",
            source_name=family_name,
            source_url=source_url,
            role=family_role,
            evidence_status="metadata_supported",
            content=(
                f"Dataset family with "
                f"{len(kept_members)} retained record(s)."
            ),
            details=details,
            provenance_note=(
                "Dataset metadata retrieved from DataCite. "
                "Relevance was evaluated only from supplied metadata."
            )
        )
    )

    evidence_counter += 1

In [95]:
for evidence in code_evidence_store:

    details = []

    for method in evidence.methods:
        details.append(
            f"Method: {method}"
        )

    for data_source in (
        evidence.datasets_or_data_sources
    ):
        details.append(
            f"Data source: {data_source}"
        )

    for framework in (
        evidence.frameworks_or_languages
    ):
        details.append(
            f"Framework/language: {framework}"
        )

    for artifact in (
        evidence.available_artifacts
    ):
        details.append(
            f"Artifact: {artifact}"
        )

    for limitation in (
        evidence.limitations_or_missing_information
    ):
        details.append(
            f"Limitation: {limitation}"
        )

    unified_evidence_store.append(
        UnifiedEvidenceItem(
            evidence_id=f"E{evidence_counter}",
            source_type="code",
            source_name=evidence.full_name,
            source_url=evidence.repository_url,
            role=evidence.role,
            evidence_status="readme_supported",
            content=evidence.evidence_summary,
            details=details,
            provenance_note=(
                "Implementation evidence extracted from "
                "the repository README and repository metadata."
            )
        )
    )

    evidence_counter += 1

In [96]:
for evidence in documentation_evidence_store:

    details = []

    for capability in evidence.capabilities:
        details.append(
            f"Capability: {capability}"
        )

    for product in evidence.data_products:
        details.append(
            f"Data product: {product}"
        )

    for access_method in evidence.access_methods:
        details.append(
            f"Access method: {access_method}"
        )

    for relevance in (
        evidence.implementation_relevance
    ):
        details.append(
            f"Implementation relevance: {relevance}"
        )

    for limitation in (
        evidence.limitations_or_missing_information
    ):
        details.append(
            f"Limitation: {limitation}"
        )

    unified_evidence_store.append(
        UnifiedEvidenceItem(
            evidence_id=f"E{evidence_counter}",
            source_type="documentation",
            source_name=evidence.source_name,
            source_url=evidence.source_url,
            role="official_reference",
            evidence_status=(
                "official_documentation_supported"
            ),
            content=evidence.evidence_summary,
            details=details,
            provenance_note=(
                f"Evidence extracted from official technical "
                f"documentation provided by {evidence.provider}."
            )
        )
    )

    evidence_counter += 1

In [97]:
print(
    f"✅ Unified evidence items: "
    f"{len(unified_evidence_store)}"
)

source_type_counts = {}

for item in unified_evidence_store:

    source_type_counts[
        item.source_type
    ] = (
        source_type_counts.get(
            item.source_type,
            0
        ) + 1
    )

print()

for source_type, count in (
    source_type_counts.items()
):
    print(
        f"{source_type}: {count}"
    )

✅ Unified evidence items: 32

paper: 21
dataset: 3
code: 5
documentation: 3


In [98]:
status_counts = {}

for item in unified_evidence_store:

    status_counts[
        item.evidence_status
    ] = (
        status_counts.get(
            item.evidence_status,
            0
        ) + 1
    )

print("Evidence status summary:")

for status, count in status_counts.items():
    print(
        f"{status}: {count}"
    )

Evidence status summary:
verified_supported: 21
metadata_supported: 3
readme_supported: 5
official_documentation_supported: 3


In [99]:
for item in unified_evidence_store:

    print("=" * 90)

    print(
        f"{item.evidence_id} | "
        f"{item.source_type.upper()}"
    )

    print(
        f"Source: {item.source_name}"
    )

    print(
        f"Role: {item.role}"
    )

    print(
        f"Status: {item.evidence_status}"
    )

    print(
        f"Evidence: {item.content[:500]}"
    )

    print()

E1 | PAPER
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Role: scientific_evidence
Status: verified_supported
Evidence: Machine learning has seen the most activity in forecasting geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flares occurrence, coronal mass ejection propagation time, and solar wind speed.

E2 | PAPER
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Role: scientific_evidence
Status: verified_supported
Evidence: The paper uses or discusses the method: machine learning

E3 | PAPER
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Role: scientific_evidence
Status: verified_supported
Evidence: The paper uses or discusses the method: physics-based and machine learning approaches (gray box)

E4 | PAPER
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Role: scientific_evidence
Status: verified_sup

## Phase 15 — Multi-Source Decision Analysis

This stage converts the unified evidence base into a technically
grounded system direction.

The Decision Agent:

- uses evidence from papers, datasets, code, and official documentation
- preserves differences in evidence status
- links recommendations to explicit evidence IDs
- distinguishes scientific support from implementation examples
- preserves uncertainty and unresolved questions
- does not treat repository implementations as scientific validation

In [100]:
class EvidenceLinkedStatement(BaseModel):
    statement: str

    evidence_ids: List[str] = Field(
        default_factory=list
    )


class MultiSourceDecision(BaseModel):
    system_direction: EvidenceLinkedStatement

    data_strategy: List[
        EvidenceLinkedStatement
    ] = Field(default_factory=list)

    baseline_strategy: List[
        EvidenceLinkedStatement
    ] = Field(default_factory=list)

    advanced_model_strategy: List[
        EvidenceLinkedStatement
    ] = Field(default_factory=list)

    evaluation_strategy: List[
        EvidenceLinkedStatement
    ] = Field(default_factory=list)

    implementation_constraints: List[
        EvidenceLinkedStatement
    ] = Field(default_factory=list)

    unresolved_questions: List[str] = Field(
        default_factory=list
    )

    decision_summary: str

In [101]:
decision_evidence_payload = []

for item in unified_evidence_store:

    decision_evidence_payload.append({
        "evidence_id": item.evidence_id,
        "source_type": item.source_type,
        "source_name": item.source_name,
        "role": item.role,
        "evidence_status": item.evidence_status,
        "content": item.content,
        "details": [
            detail[:800]
            for detail in item.details[:10]
        ]
    })

print(
    f"Evidence items prepared for decision: "
    f"{len(decision_evidence_payload)}"
)

Evidence items prepared for decision: 32


In [102]:
def create_multisource_decision(
    user_question: str,
    evidence_store: list,
    scientific_analysis: AnalysisResult,
    cross_source_report: CrossSourceReport
):
    evidence_payload = []

    for item in evidence_store:

        evidence_payload.append({
            "evidence_id": item.evidence_id,
            "source_type": item.source_type,
            "source_name": item.source_name,
            "role": item.role,
            "evidence_status": item.evidence_status,
            "content": item.content,
            "details": [
                detail[:800]
                for detail in item.details[:10]
            ]
        })

    evidence_text = json.dumps(
        evidence_payload,
        indent=2,
        ensure_ascii=False
    )

    analysis_text = json.dumps(
        scientific_analysis.model_dump(),
        indent=2,
        ensure_ascii=False
    )

    relations_text = json.dumps(
        cross_source_report.model_dump(),
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Multi-Source Decision Agent of AURA.

USER RESEARCH QUESTION:

{user_question}

SCIENTIFIC SYNTHESIS:

{analysis_text}

CROSS-SOURCE SCIENTIFIC RELATIONSHIPS:

{relations_text}

UNIFIED EVIDENCE STORE:

{evidence_text}

Your task is to propose an evidence-grounded technical direction
for implementing the requested solar flare prediction system.

Important evidence rules:

1. Use ONLY the supplied evidence.
2. Do not use outside knowledge.
3. Every evidence ID you cite must exist in the supplied evidence store.
4. Do not invent datasets, architectures, APIs, metrics, or capabilities.
5. Scientific paper evidence and implementation evidence have different roles.
6. Repository README evidence demonstrates documented implementation,
   not scientific superiority or validation.
7. Dataset metadata demonstrates documented availability and contents,
   not proven dataset quality.
8. Official documentation demonstrates documented access capabilities,
   not predictive model performance.
9. Prefer recommendations supported by multiple source types when possible.
10. Do not call an approach "best", "optimal", or superior unless the
    supplied evidence explicitly supports that conclusion.
11. Preserve the probabilistic nature and uncertainty of solar flare
    forecasting described in the scientific evidence.
12. Preserve the need for rigorous training and testing.
13. Separate a practical baseline from more advanced implementation paths.
14. If the supplied evidence is insufficient to decide something,
    place it in unresolved_questions.
15. Every recommendation statement must include supporting evidence IDs.

Return ONLY valid JSON with exactly this structure:

{{
    "system_direction": {{
        "statement": "...",
        "evidence_ids": ["E1", "E19"]
    }},

    "data_strategy": [
        {{
            "statement": "...",
            "evidence_ids": ["E19", "E20"]
        }}
    ],

    "baseline_strategy": [
        {{
            "statement": "...",
            "evidence_ids": ["E11", "E23"]
        }}
    ],

    "advanced_model_strategy": [
        {{
            "statement": "...",
            "evidence_ids": ["E22"]
        }}
    ],

    "evaluation_strategy": [
        {{
            "statement": "...",
            "evidence_ids": ["E12", "E18"]
        }}
    ],

    "implementation_constraints": [
        {{
            "statement": "...",
            "evidence_ids": ["E27", "E28"]
        }}
    ],

    "unresolved_questions": [
        "..."
    ],

    "decision_summary": "..."
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=DEFAULT_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return MultiSourceDecision(
        **result
    )

In [103]:
decision_result = create_multisource_decision(
    question,
    unified_evidence_store,
    analysis_result,
    cross_source_report
)

print(
    json.dumps(
        decision_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash
✅ Success with: gemini-3.5-flash
{
  "system_direction": {
    "statement": "Develop a gray-box solar flare prediction system that integrates physics-based properties with machine learning, adopting an inherently probabilistic forecasting paradigm focused on the reliable assessment of uncertainties to address the barrier of stochasticity in space weather events.",
    "evidence_ids": [
      "E3",
      "E4",
      "E5",
      "E17",
      "E21"
    ]
  },
  "data_strategy": [
    {
      "statement": "Utilize SunPy as the open-source Python data container and analysis environment to search, download, and coordinate solar observations across multiple data providers.",
      "evidence_ids": [
        "E30"
      ]
    },
    {
      "statement": "Integrate multi-modal spatial and temporal data, specifically Helioseismic and Magnetic Imager (HMI) magnetograms, Atmospheric Imaging Assembly (AIA) cutouts from SDO, and 1-minute averaged GOES soft X-ray flux ob

In [104]:
def validate_decision_evidence_ids(
    decision: MultiSourceDecision,
    evidence_store: list
):
    valid_ids = {
        item.evidence_id
        for item in evidence_store
    }

    linked_statements = [
        decision.system_direction
    ]

    linked_statements.extend(
        decision.data_strategy
    )

    linked_statements.extend(
        decision.baseline_strategy
    )

    linked_statements.extend(
        decision.advanced_model_strategy
    )

    linked_statements.extend(
        decision.evaluation_strategy
    )

    linked_statements.extend(
        decision.implementation_constraints
    )

    referenced_ids = []

    for item in linked_statements:
        referenced_ids.extend(
            item.evidence_ids
        )

    invalid_ids = sorted(
        {
            evidence_id
            for evidence_id
            in referenced_ids
            if evidence_id not in valid_ids
        }
    )

    return {
        "valid_reference_count": len([
            evidence_id
            for evidence_id in referenced_ids
            if evidence_id in valid_ids
        ]),
        "invalid_evidence_ids": invalid_ids
    }

In [105]:
decision_validation = (
    validate_decision_evidence_ids(
        decision_result,
        unified_evidence_store
    )
)

print(
    json.dumps(
        decision_validation,
        indent=2,
        ensure_ascii=False
    )
)

{
  "valid_reference_count": 26,
  "invalid_evidence_ids": []
}


In [106]:
print("SYSTEM DIRECTION")
print(
    decision_result
    .system_direction
    .statement
)

print()

print("BASELINE STRATEGY")
for item in decision_result.baseline_strategy:
    print(
        f"- {item.statement} "
        f"{item.evidence_ids}"
    )

print()

print("ADVANCED MODEL STRATEGY")
for item in decision_result.advanced_model_strategy:
    print(
        f"- {item.statement} "
        f"{item.evidence_ids}"
    )

print()

print("UNRESOLVED QUESTIONS")
for item in decision_result.unresolved_questions:
    print(f"- {item}")

SYSTEM DIRECTION
Develop a gray-box solar flare prediction system that integrates physics-based properties with machine learning, adopting an inherently probabilistic forecasting paradigm focused on the reliable assessment of uncertainties to address the barrier of stochasticity in space weather events.

BASELINE STRATEGY
- Establish a Python-based machine learning baseline using Scikit-Learn to model solar flares from NASA SDO/HMI magnetograms and magnetic feature properties, such as those generated by SMART. ['E27', 'E28']
- Implement time-series feature engineering on physical solar parameters to address the historical limitations of models that ignore time dependence on features. ['E25']
- Employ a random forest regression model using standard scientific Python packages (NumPy, SciPy, pandas, and scikit-learn) with GOES/XRS data to establish a predictive baseline for flare dynamics such as remaining flare duration. ['E29']

ADVANCED MODEL STRATEGY
- Deploy an end-to-end multi-modal

### Phase 15.5 — Decision Evidence Audit

This stage verifies that each decision statement is semantically
supported by its cited evidence.

The audit:

- checks claim-to-evidence alignment
- detects overgeneralization
- detects unsupported implementation details
- proposes narrower evidence-grounded wording
- preserves valid evidence IDs

In [107]:
class DecisionStatementAudit(BaseModel):
    section: str
    item_index: int

    status: str

    issues: List[str] = Field(
        default_factory=list
    )

    revised_statement: str


class DecisionAuditReport(BaseModel):
    audits: List[
        DecisionStatementAudit
    ] = Field(default_factory=list)

    summary: str

In [108]:
def prepare_decision_statements(
    decision: MultiSourceDecision
):
    statements = []

    statements.append({
        "section": "system_direction",
        "item_index": 0,
        "statement": (
            decision
            .system_direction
            .statement
        ),
        "evidence_ids": (
            decision
            .system_direction
            .evidence_ids
        )
    })

    sections = {
        "data_strategy":
            decision.data_strategy,

        "baseline_strategy":
            decision.baseline_strategy,

        "advanced_model_strategy":
            decision.advanced_model_strategy,

        "evaluation_strategy":
            decision.evaluation_strategy,

        "implementation_constraints":
            decision.implementation_constraints
    }

    for section_name, items in sections.items():

        for index, item in enumerate(items):

            statements.append({
                "section": section_name,
                "item_index": index,
                "statement": item.statement,
                "evidence_ids": item.evidence_ids
            })

    return statements

In [109]:
decision_evidence_lookup = {
    item.evidence_id: {
        "source_type": item.source_type,
        "source_name": item.source_name,
        "evidence_status": item.evidence_status,
        "content": item.content,
        "details": item.details
    }
    for item in unified_evidence_store
}

In [110]:
def audit_decision_evidence(
    decision: MultiSourceDecision,
    evidence_store: list
):
    statements = prepare_decision_statements(
        decision
    )

    evidence_lookup = {
        item.evidence_id: {
            "source_type": item.source_type,
            "source_name": item.source_name,
            "evidence_status": item.evidence_status,
            "content": item.content,
            "details": item.details
        }
        for item in evidence_store
    }

    audit_payload = []

    for statement in statements:

        cited_evidence = {}

        for evidence_id in (
            statement["evidence_ids"]
        ):

            if evidence_id in evidence_lookup:
                cited_evidence[
                    evidence_id
                ] = evidence_lookup[
                    evidence_id
                ]

        audit_payload.append({
            **statement,
            "cited_evidence":
                cited_evidence
        })

    audit_text = json.dumps(
        audit_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Evidence Entailment Auditor of AURA.

Your task is to verify whether each technical decision statement
is actually supported by its cited evidence.

DECISION STATEMENTS AND CITED EVIDENCE:

{audit_text}

For every statement assign exactly one status:

- supported
- partially_supported
- unsupported

Definitions:

supported:
Every material technical claim in the statement is justified
by the cited evidence.

partially_supported:
The general direction is supported, but some wording,
scope, implementation detail, or causal claim goes beyond
the evidence.

unsupported:
The central recommendation is not justified by the cited evidence.

Rules:

1. Use ONLY the supplied cited evidence.
2. Do not use outside knowledge.
3. Be strict about scope.
4. README evidence documents an implementation example,
   not scientific superiority.
5. Dataset metadata documents availability and contents,
   not predictive effectiveness.
6. Official documentation supports documented data access
   capabilities, not model performance.
7. Do not infer classification from regression evidence.
8. Do not infer data leakage unless the evidence explicitly
   discusses data leakage.
9. Do not turn association or clues into causal claims.
10. If wording is too broad, provide a minimally revised,
    evidence-grounded statement.
11. If the statement is fully supported, revised_statement
    should preserve the original meaning.
12. Every item in issues must be a plain string.

Return ONLY valid JSON:

{{
    "audits": [
        {{
            "section": "baseline_strategy",
            "item_index": 0,
            "status": "partially_supported",
            "issues": [
                "..."
            ],
            "revised_statement": "..."
        }}
    ],
    "summary": "..."
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=DEFAULT_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return DecisionAuditReport(
        **result
    )

In [111]:
decision_audit = audit_decision_evidence(
    decision_result,
    unified_evidence_store
)

print(
    json.dumps(
        decision_audit.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash
⚠️ gemini-3.5-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.6-flash
⚠️ gemini-3.6-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.7-flash
⚠️ gemini-3.7-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "audits": [
    {
      "section": "system_direction",
      "item_index": 0,
      "status": "supported",
      "issues": [],
      "revised_statement": "Develop a gray-box solar flare prediction system that integrates physics-based properties with machine learning, adopting an inherently probabilistic forecasting paradigm focused on the reliable assessment of uncertainties to address the barrier of stochasticity in space weather events."
    },
    {
      "section": "data_strategy",
      "item_index": 0,
      "status": "supported",
      "issues": [],
      "revised_statement": "Utilize Su

In [112]:
audit_counts = {
    "supported": 0,
    "partially_supported": 0,
    "unsupported": 0
}

for audit in decision_audit.audits:

    if audit.status in audit_counts:
        audit_counts[
            audit.status
        ] += 1

print("Decision evidence audit:")

for status, count in audit_counts.items():
    print(
        f"{status}: {count}"
    )

Decision evidence audit:
supported: 15
partially_supported: 1
unsupported: 0


In [113]:
for audit in decision_audit.audits:

    if audit.status != "supported":

        print("=" * 90)
        print(
            f"{audit.section} "
            f"[{audit.item_index}]"
        )
        print(
            f"Status: {audit.status}"
        )

        for issue in audit.issues:
            print(f"- {issue}")

        print()
        print("REVISED:")
        print(
            audit.revised_statement
        )
        print()

advanced_model_strategy [2]
Status: partially_supported
- Dataset metadata documents availability and contents, but does not provide details on specific integrated gradients model attribution implementations.

REVISED:
Incorporate a lightweight and interpretable sequence modeling architecture, specifically utilizing datasets associated with Mamba-based feature and temporal attribution via integrated gradients.



### Phase 15.6 — Audited Decision

The original multi-source decision is updated using the semantic
evidence audit.

Partially supported statements are replaced with narrower,
evidence-grounded wording before implementation planning begins.

In [114]:
def apply_decision_audit(
    decision: MultiSourceDecision,
    audit_report: DecisionAuditReport
):
    audited_decision = decision.model_copy(
        deep=True
    )

    for audit in audit_report.audits:

        revised_statement = (
            audit.revised_statement.strip()
        )

        if not revised_statement:
            continue

        if audit.section == "system_direction":

            audited_decision.system_direction.statement = (
                revised_statement
            )

            continue

        section_items = getattr(
            audited_decision,
            audit.section,
            None
        )

        if section_items is None:
            continue

        if (
            0 <= audit.item_index
            < len(section_items)
        ):
            section_items[
                audit.item_index
            ].statement = revised_statement

    return audited_decision

In [115]:
audited_decision_result = apply_decision_audit(
    decision_result,
    decision_audit
)

print(
    json.dumps(
        audited_decision_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

{
  "system_direction": {
    "statement": "Develop a gray-box solar flare prediction system that integrates physics-based properties with machine learning, adopting an inherently probabilistic forecasting paradigm focused on the reliable assessment of uncertainties to address the barrier of stochasticity in space weather events.",
    "evidence_ids": [
      "E3",
      "E4",
      "E5",
      "E17",
      "E21"
    ]
  },
  "data_strategy": [
    {
      "statement": "Utilize SunPy as the open-source Python data container and analysis environment to search, download, and coordinate solar observations across multiple data providers.",
      "evidence_ids": [
        "E30"
      ]
    },
    {
      "statement": "Integrate multi-modal spatial and temporal data, specifically Helioseismic and Magnetic Imager (HMI) magnetograms, Atmospheric Imaging Assembly (AIA) cutouts from SDO, and 1-minute averaged GOES soft X-ray flux observations (0.1–0.8 nm and 0.05–0.4 nm passbands).",
      "evid

In [116]:
print("AUDITED BASELINE STRATEGY")

for item in (
    audited_decision_result
    .baseline_strategy
):
    print(
        f"- {item.statement} "
        f"{item.evidence_ids}"
    )

print()

print("AUDITED EVALUATION STRATEGY")

for item in (
    audited_decision_result
    .evaluation_strategy
):
    print(
        f"- {item.statement} "
        f"{item.evidence_ids}"
    )

AUDITED BASELINE STRATEGY
- Establish a Python-based machine learning baseline using Scikit-Learn to model solar flares from NASA SDO/HMI magnetograms and magnetic feature properties, such as those generated by SMART. ['E27', 'E28']
- Implement time-series feature engineering on physical solar parameters to address the historical limitations of models that ignore time dependence on features. ['E25']
- Employ a random forest regression model using standard scientific Python packages (NumPy, SciPy, pandas, and scikit-learn) with GOES/XRS data to establish a predictive baseline for flare dynamics such as remaining flare duration. ['E29']

AUDITED EVALUATION STRATEGY
- Enforce highly rigorous training and testing protocols to prevent overly optimistic pre-operational prediction performance when moving from retrospective validation to operational deployment. ['E15', 'E21']
- Evaluate forecasting models using structured 10-fold cross-validation datasets (10CV) mapped to NOAA active region nu

## Phase 16 — Evidence-Grounded Implementation Roadmap

This stage converts the audited technical decision into an actionable
implementation roadmap.

The roadmap distinguishes between:

- evidence-backed technical directions
- engineering scaffolding required to implement the system

It preserves evidence provenance and does not present engineering
choices as scientifically validated conclusions.

In [117]:
class RoadmapStep(BaseModel):
    step_id: str
    phase: str
    objective: str

    actions: List[str] = Field(
        default_factory=list
    )

    basis_type: str

    evidence_ids: List[str] = Field(
        default_factory=list
    )

    deliverable: str

    validation_criteria: List[str] = Field(
        default_factory=list
    )

    dependencies: List[str] = Field(
        default_factory=list
    )


class ImplementationRoadmap(BaseModel):
    steps: List[RoadmapStep] = Field(
        default_factory=list
    )

    unresolved_research_questions: List[str] = Field(
        default_factory=list
    )

    engineering_assumptions: List[str] = Field(
        default_factory=list
    )

    final_target: str

In [118]:
def create_implementation_roadmap(
    user_question: str,
    audited_decision: MultiSourceDecision,
    evidence_store: list
):
    decision_text = json.dumps(
        audited_decision.model_dump(),
        indent=2,
        ensure_ascii=False
    )

    evidence_payload = [
        {
            "evidence_id": item.evidence_id,
            "source_type": item.source_type,
            "source_name": item.source_name,
            "evidence_status": item.evidence_status,
            "content": item.content,
            "details": [
                detail[:700]
                for detail in item.details[:8]
            ]
        }
        for item in evidence_store
    ]

    evidence_text = json.dumps(
        evidence_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Implementation Roadmap Agent of AURA.

USER RESEARCH QUESTION:

{user_question}

AUDITED MULTI-SOURCE DECISION:

{decision_text}

UNIFIED EVIDENCE STORE:

{evidence_text}

Create a chronological implementation roadmap for the proposed
solar flare forecasting system.

Important rules:

1. Use the AUDITED decision as the primary technical direction.

2. Use ONLY the supplied evidence when making scientific or
   source-specific claims.

3. Do not reintroduce wording removed by the evidence audit.

4. Do not claim Random Forest occurrence classification from
   GOES/XRS evidence. The supplied GOES/XRS Random Forest example
   supports flare-duration regression.

5. Do not treat flare-duration forecasting as equivalent to
   flare-occurrence forecasting.

6. Do not introduce specific accuracy targets, benchmark scores,
   thresholds, or performance improvements unless supplied by evidence.

7. Do not invent APIs, datasets, model architectures, or measurements.

8. Preserve the probabilistic nature of solar flare forecasting.

9. Preserve the requirement for rigorous training and testing.

10. A roadmap step may use one of exactly two basis_type values:

    - evidence_backed
    - engineering_scaffold

11. For evidence_backed steps, include valid supporting evidence IDs.

12. Engineering scaffolding may contain implementation tasks that are
    necessary to build the system but are not scientifically validated.
    These must use basis_type="engineering_scaffold".

13. Do not present engineering scaffolding as scientific evidence.

14. Keep unresolved scientific questions unresolved unless the evidence
    actually answers them.

15. Include a practical baseline before advanced multimodal models.

16. Treat the transformer implementation documented in E22 as an
    implementation example/path, not evidence that transformers are
    scientifically superior.

17. Account for documented GOES calibration differences and data
    dropouts where relevant.

18. Return between 6 and 9 chronological roadmap steps.

19. Every actions, validation_criteria, dependencies, assumptions,
    and unresolved question item must be a plain string.

Return ONLY valid JSON:

{{
    "steps": [
        {{
            "step_id": "R1",
            "phase": "...",
            "objective": "...",
            "actions": [
                "..."
            ],
            "basis_type": "evidence_backed",
            "evidence_ids": [
                "E19",
                "E27"
            ],
            "deliverable": "...",
            "validation_criteria": [
                "..."
            ],
            "dependencies": []
        }}
    ],

    "unresolved_research_questions": [
        "..."
    ],

    "engineering_assumptions": [
        "..."
    ],

    "final_target": "..."
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=DEFAULT_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return ImplementationRoadmap(
        **result
    )

In [119]:
roadmap_result = create_implementation_roadmap(
    question,
    audited_decision_result,
    unified_evidence_store
)

print(
    json.dumps(
        roadmap_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash
⚠️ gemini-3.5-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.6-flash
⚠️ gemini-3.6-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.7-flash
⚠️ gemini-3.7-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "steps": [
    {
      "step_id": "R1",
      "phase": "Data Acquisition and Environment Setup",
      "objective": "Establish the open-source Python data container and analysis environment to search, download, and coordinate solar observations across multiple data providers.",
      "actions": [
        "Utilize SunPy as the open-source Python data container and analysis environment to search, download, and coordinate solar observations.",
        "Access SDO data products including AIA, HMI, and EVE science and near-real-time data using web interfaces, FTP/Wget command-line automation, VSO, a

In [120]:
def validate_roadmap_evidence_ids(
    roadmap: ImplementationRoadmap,
    evidence_store: list
):
    valid_ids = {
        item.evidence_id
        for item in evidence_store
    }

    referenced_ids = []

    missing_evidence_steps = []

    for step in roadmap.steps:

        referenced_ids.extend(
            step.evidence_ids
        )

        if (
            step.basis_type == "evidence_backed"
            and not step.evidence_ids
        ):
            missing_evidence_steps.append(
                step.step_id
            )

    invalid_ids = sorted({
        evidence_id
        for evidence_id in referenced_ids
        if evidence_id not in valid_ids
    })

    return {
        "roadmap_steps": len(
            roadmap.steps
        ),
        "valid_reference_count": len([
            evidence_id
            for evidence_id in referenced_ids
            if evidence_id in valid_ids
        ]),
        "invalid_evidence_ids": invalid_ids,
        "evidence_backed_steps_without_evidence":
            missing_evidence_steps
    }

In [121]:
roadmap_validation = (
    validate_roadmap_evidence_ids(
        roadmap_result,
        unified_evidence_store
    )
)

print(
    json.dumps(
        roadmap_validation,
        indent=2,
        ensure_ascii=False
    )
)

{
  "roadmap_steps": 6,
  "valid_reference_count": 21,
  "invalid_evidence_ids": [],
  "evidence_backed_steps_without_evidence": []
}


In [122]:
for step in roadmap_result.steps:

    print("=" * 90)

    print(
        f"{step.step_id} — "
        f"{step.phase}"
    )

    print(
        f"Objective: "
        f"{step.objective}"
    )

    print(
        f"Basis: "
        f"{step.basis_type}"
    )

    print(
        f"Evidence: "
        f"{step.evidence_ids}"
    )

    print("Actions:")

    for action in step.actions:
        print(f"- {action}")

    print(
        f"Deliverable: "
        f"{step.deliverable}"
    )

    print("Validation:")

    for criterion in (
        step.validation_criteria
    ):
        print(f"- {criterion}")

    print()

R1 — Data Acquisition and Environment Setup
Objective: Establish the open-source Python data container and analysis environment to search, download, and coordinate solar observations across multiple data providers.
Basis: evidence_backed
Evidence: ['E30', 'E31', 'E32']
Actions:
- Utilize SunPy as the open-source Python data container and analysis environment to search, download, and coordinate solar observations.
- Access SDO data products including AIA, HMI, and EVE science and near-real-time data using web interfaces, FTP/Wget command-line automation, VSO, and SolarSoft IDL tools.
- Retrieve GOES X-ray flux observations (1-8 Å and 0.5-4.0 Å passbands, or 0.1-0.8 nm and 0.05-0.4 nm) via JSON endpoints and archive file services, while accounting for known calibration shifts across satellite generations.
Deliverable: Configured SunPy data ingestion pipeline capable of retrieving SDOcutouts, HMI magnetograms, and GOES X-ray flux observations.
Validation:
- Verify successful connection an

### Phase 16.5 — Roadmap Semantic Audit

This stage verifies that each roadmap step remains within the scope
of its cited evidence and correctly separates evidence-backed
technical direction from engineering scaffolding.

The audit detects:

- unsupported implementation details
- reintroduced claims removed by the decision audit
- overly specific validation criteria
- unsupported causal or performance claims
- incorrect evidence/scaffolding classification

In [123]:
class RoadmapStepAudit(BaseModel):
    step_id: str

    status: str

    issues: List[str] = Field(
        default_factory=list
    )

    revised_basis_type: str

    revised_objective: str

    revised_actions: List[str] = Field(
        default_factory=list
    )

    revised_deliverable: str

    revised_validation_criteria: List[str] = Field(
        default_factory=list
    )


class RoadmapAuditReport(BaseModel):
    audits: List[
        RoadmapStepAudit
    ] = Field(default_factory=list)

    summary: str

In [124]:
def audit_implementation_roadmap(
    roadmap: ImplementationRoadmap,
    evidence_store: list,
    audited_decision: MultiSourceDecision
):
    evidence_lookup = {
        item.evidence_id: {
            "source_type": item.source_type,
            "source_name": item.source_name,
            "evidence_status": item.evidence_status,
            "content": item.content,
            "details": item.details
        }
        for item in evidence_store
    }

    roadmap_payload = []

    for step in roadmap.steps:

        cited_evidence = {}

        for evidence_id in step.evidence_ids:

            if evidence_id in evidence_lookup:
                cited_evidence[evidence_id] = (
                    evidence_lookup[evidence_id]
                )

        roadmap_payload.append({
            "step": step.model_dump(),
            "cited_evidence": cited_evidence
        })

    roadmap_text = json.dumps(
        roadmap_payload,
        indent=2,
        ensure_ascii=False
    )

    decision_text = json.dumps(
        audited_decision.model_dump(),
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Roadmap Evidence Auditor of AURA.

AUDITED DECISION:

{decision_text}

ROADMAP STEPS AND THEIR CITED EVIDENCE:

{roadmap_text}

Audit every roadmap step for evidence alignment.

For each step assign exactly one status:

- supported
- partially_supported
- unsupported

Important rules:

1. Use ONLY the supplied decision and cited evidence.

2. Do not use outside knowledge.

3. Do not allow claims removed by the previous decision audit
   to reappear.

4. Do not infer data leakage unless explicitly documented.

5. Do not claim calibration correction methods when evidence
   only documents calibration differences.

6. Do not treat containerization, orchestration, scheduling,
   refactoring, module design, or deployment architecture as
   scientifically evidence-backed unless explicitly documented.

7. Engineering implementation choices may be retained, but they
   must use revised_basis_type="engineering_scaffold".

8. Do not infer that 170+ FLARECAST predictors can automatically
   be extracted from every selected dataset.

9. Do not invent specific evaluation products such as reliability
   diagrams, uncertainty intervals, or benchmark thresholds unless
   explicitly supported by evidence.

10. Do not use "stable convergence", "robust", "guarantee",
    or similar performance language unless explicitly supported.

11. GOES calibration differences may require handling, but do not
    invent a correction method.

12. Preserve Random Forest regression as flare-duration forecasting;
    do not convert it into flare-occurrence classification.

13. Preserve the probabilistic forecasting direction.

14. revised_basis_type must be exactly:
    - evidence_backed
    - engineering_scaffold

15. Keep revisions minimal and practical.

16. Every list item must be a plain string.

Return ONLY valid JSON:

{{
    "audits": [
        {{
            "step_id": "R1",
            "status": "partially_supported",
            "issues": [
                "..."
            ],
            "revised_basis_type": "engineering_scaffold",
            "revised_objective": "...",
            "revised_actions": [
                "..."
            ],
            "revised_deliverable": "...",
            "revised_validation_criteria": [
                "..."
            ]
        }}
    ],
    "summary": "..."
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=DEFAULT_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return RoadmapAuditReport(
        **result
    )

In [125]:
roadmap_audit = audit_implementation_roadmap(
    roadmap_result,
    unified_evidence_store,
    audited_decision_result
)

print(
    json.dumps(
        roadmap_audit.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash
⚠️ gemini-3.5-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.6-flash
⚠️ gemini-3.6-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.7-flash
⚠️ gemini-3.7-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "audits": [
    {
      "step_id": "R1",
      "status": "partially_supported",
      "issues": [
        "Action item regarding GOES calibration mentions handling calibration shifts, but rule 5 and rule 11 state that GOES calibration differences may be noted, but no specific correction method should be invented."
      ],
      "revised_basis_type": "engineering_scaffold",
      "revised_objective": "Establish the open-source Python data container and analysis environment to search, download, and coordinate solar observations across multiple data providers.",
      "revised_actions": [
       

In [126]:
roadmap_audit_counts = {
    "supported": 0,
    "partially_supported": 0,
    "unsupported": 0
}

for audit in roadmap_audit.audits:

    if audit.status in roadmap_audit_counts:
        roadmap_audit_counts[
            audit.status
        ] += 1

print("Roadmap evidence audit:")

for status, count in (
    roadmap_audit_counts.items()
):
    print(
        f"{status}: {count}"
    )

Roadmap evidence audit:
supported: 5
partially_supported: 1
unsupported: 0


In [127]:
for audit in roadmap_audit.audits:

    if audit.status != "supported":

        print("=" * 90)
        print(audit.step_id)
        print(
            f"Status: {audit.status}"
        )
        print(
            f"Revised basis: "
            f"{audit.revised_basis_type}"
        )

        for issue in audit.issues:
            print(f"- {issue}")

        print()
        print("REVISED OBJECTIVE:")
        print(
            audit.revised_objective
        )

        print()
        print("REVISED ACTIONS:")

        for action in audit.revised_actions:
            print(f"- {action}")

        print()

R1
Status: partially_supported
Revised basis: engineering_scaffold
- Action item regarding GOES calibration mentions handling calibration shifts, but rule 5 and rule 11 state that GOES calibration differences may be noted, but no specific correction method should be invented.

REVISED OBJECTIVE:
Establish the open-source Python data container and analysis environment to search, download, and coordinate solar observations across multiple data providers.

REVISED ACTIONS:
- Utilize SunPy as the open-source Python data container and analysis environment to search, download, and coordinate solar observations.
- Access SDO data products including AIA, HMI, and EVE science and near-real-time data using web interfaces, FTP/Wget command-line automation, VSO, and SolarSoft IDL tools.
- Retrieve GOES X-ray flux observations (1-8 Å and 0.5-4.0 Å passbands, or 0.1-0.8 nm and 0.05-0.4 nm) via JSON endpoints and archive file services, noting known calibration differences across satellite generations

### Phase 16.6 — Audited Roadmap

The roadmap audit is applied before final reporting.

A deterministic guardrail layer then checks for known forms of
evidence overreach that may be missed by the language-model auditor.

In [128]:
def apply_roadmap_audit(
    roadmap: ImplementationRoadmap,
    audit_report: RoadmapAuditReport
):
    audited_roadmap = roadmap.model_copy(
        deep=True
    )

    audit_lookup = {
        audit.step_id: audit
        for audit in audit_report.audits
    }

    for step in audited_roadmap.steps:

        audit = audit_lookup.get(
            step.step_id
        )

        if audit is None:
            continue

        step.basis_type = (
            audit.revised_basis_type
        )

        step.objective = (
            audit.revised_objective
        )

        step.actions = list(
            audit.revised_actions
        )

        step.deliverable = (
            audit.revised_deliverable
        )

        step.validation_criteria = list(
            audit.revised_validation_criteria
        )

    return audited_roadmap

In [129]:
audited_roadmap_result = apply_roadmap_audit(
    roadmap_result,
    roadmap_audit
)

print(
    f"Audited roadmap steps: "
    f"{len(audited_roadmap_result.steps)}"
)

Audited roadmap steps: 6


In [130]:
audited_step_lookup = {
    step.step_id: step
    for step in audited_roadmap_result.steps
}

In [131]:
r3 = audited_step_lookup["R3"]

r3.validation_criteria = [
    (
        "Document the physical parameter representations "
        "and the GOES-generation calibration differences "
        "considered during data preparation."
    )
]

In [132]:
r4 = audited_step_lookup["R4"]

r4.validation_criteria = [
    (
        "Verify that the tabular baseline and the "
        "Random Forest flare-duration regressor execute "
        "on the designated evaluation data and produce outputs."
    )
]

In [133]:
r6 = audited_step_lookup["R6"]

r6.actions[0] = (
    "Use rigorously separated training and testing data "
    "to avoid overly optimistic pre-operational "
    "prediction performance."
)

r6.deliverable = (
    "An evaluation module reporting probabilistic outputs, "
    "uncertainty assessments, and eruptive-flare "
    "transition tracking."
)

r6.validation_criteria = [
    (
        "Evaluate probabilistic outputs and uncertainty "
        "assessments on rigorously separated test data."
    )
]

In [138]:
r7 = audited_step_lookup.get("R7")

if r7 is not None:

    if r7.actions:
        r7.actions[0] = (
            "Containerize the baseline and transformer "
            "model pipelines using Docker."
        )

    print("✅ R7 patch applied.")

else:
    print(
        "ℹ️ R7 does not exist in this roadmap. "
        "Patch skipped."
    )

ℹ️ R7 does not exist in this roadmap. Patch skipped.


In [142]:
def validate_final_roadmap(
    roadmap: ImplementationRoadmap,
    evidence_store: list
):
    valid_evidence_ids = {
        item.evidence_id
        for item in evidence_store
    }

    valid_basis_types = {
        "evidence_backed",
        "engineering_scaffold"
    }

    step_ids = [
        step.step_id
        for step in roadmap.steps
    ]

    invalid_evidence_ids = []
    invalid_basis_types = []
    missing_evidence = []
    invalid_dependencies = []
    overreach_flags = []

    forbidden_phrases = [
        "data leakage",
        "calibration corrections",
        "stable convergence",
        "guarantee",
        "reliability diagrams",
        "uncertainty intervals",
        "temporal or active-region splits"
    ]

    for step in roadmap.steps:

        if (
            step.basis_type
            not in valid_basis_types
        ):
            invalid_basis_types.append(
                step.step_id
            )

        if (
            step.basis_type
            == "evidence_backed"
            and not step.evidence_ids
        ):
            missing_evidence.append(
                step.step_id
            )

        for evidence_id in step.evidence_ids:

            if (
                evidence_id
                not in valid_evidence_ids
            ):
                invalid_evidence_ids.append(
                    evidence_id
                )

        for dependency in step.dependencies:

            if dependency not in step_ids:
                invalid_dependencies.append({
                    "step": step.step_id,
                    "dependency": dependency
                })

        combined_text = " ".join([
            step.objective,
            *step.actions,
            step.deliverable,
            *step.validation_criteria
        ]).lower()

        for phrase in forbidden_phrases:

            if phrase.lower() in combined_text:
                overreach_flags.append({
                    "step": step.step_id,
                    "phrase": phrase
                })

    duplicate_step_ids = sorted({
        step_id
        for step_id in step_ids
        if step_ids.count(step_id) > 1
    })

    return {
        "roadmap_steps": len(
            roadmap.steps
        ),

        "invalid_evidence_ids": sorted(
            set(invalid_evidence_ids)
        ),

        "invalid_basis_types": sorted(
            set(invalid_basis_types)
        ),

        "evidence_backed_steps_without_evidence":
            sorted(
                set(missing_evidence)
            ),

        "invalid_dependencies":
            invalid_dependencies,

        "duplicate_step_ids":
            duplicate_step_ids,

        "overreach_flags":
            overreach_flags
    }

In [144]:
final_roadmap_validation = (
    validate_final_roadmap(
        audited_roadmap_result,
        unified_evidence_store
    )
)

print(
    json.dumps(
        final_roadmap_validation,
        indent=2,
        ensure_ascii=False
    )
)

{
  "roadmap_steps": 6,
  "invalid_evidence_ids": [],
  "invalid_basis_types": [],
  "evidence_backed_steps_without_evidence": [],
  "invalid_dependencies": [],
  "duplicate_step_ids": [],
  "overreach_flags": []
}


## Phase 17 — Final Traceable Multi-Source Report

This stage generates the final AURA research report.

The report combines:

- the original research question
- scientific evidence synthesis
- cross-source scientific relationships
- multi-source evidence coverage
- audited technical decisions
- audited implementation roadmap
- unresolved research questions
- engineering assumptions
- evidence provenance

Every evidence-backed recommendation remains traceable to explicit
evidence IDs in the unified evidence store.

In [146]:
def markdown_bullets(items):
    if not items:
        return "- None identified."

    return "\n".join(
        f"- {item}"
        for item in items
    )


def evidence_ids_text(ids):
    if not ids:
        return "Engineering scaffold"

    return ", ".join(ids)

In [148]:
def build_final_aura_report(
    question,
    analysis_result,
    cross_source_report,
    audited_decision,
    audited_roadmap,
    evidence_store,
    decision_audit,
    roadmap_audit,
    final_roadmap_validation
):
    source_type_counts = {}
    status_counts = {}

    for item in evidence_store:

        source_type_counts[
            item.source_type
        ] = (
            source_type_counts.get(
                item.source_type,
                0
            ) + 1
        )

        status_counts[
            item.evidence_status
        ] = (
            status_counts.get(
                item.evidence_status,
                0
            ) + 1
        )

    report = []

    report.append(
        "# AURA — Scientific & Technical Research Report"
    )

    report.append(
        "\n## Research Question\n"
    )

    report.append(
        question.strip()
    )

    report.append(
        "\n## Evidence Coverage\n"
    )

    report.append(
        f"- Total unified evidence items: "
        f"{len(evidence_store)}"
    )

    for source_type, count in (
        source_type_counts.items()
    ):
        report.append(
            f"- {source_type.title()}: {count}"
        )

    report.append(
        "\n### Evidence Status\n"
    )

    for status, count in status_counts.items():
        report.append(
            f"- {status}: {count}"
        )

    # ------------------------------------------------
    # Scientific synthesis
    # ------------------------------------------------

    report.append(
        "\n## Scientific Evidence Synthesis\n"
    )

    report.append(
        analysis_result.synthesis
    )

    report.append(
        "\n### Key Findings\n"
    )

    report.append(
        markdown_bullets(
            analysis_result.key_findings
        )
    )

    report.append(
        "\n### Methods Identified\n"
    )

    report.append(
        markdown_bullets(
            analysis_result.methods_identified
        )
    )

    report.append(
        "\n### Data Sources Identified\n"
    )

    report.append(
        markdown_bullets(
            analysis_result.data_sources_identified
        )
    )

    report.append(
        "\n### Methodological Patterns\n"
    )

    report.append(
        markdown_bullets(
            analysis_result.methodological_patterns
        )
    )

    report.append(
        "\n### Scientific Uncertainties\n"
    )

    report.append(
        markdown_bullets(
            analysis_result.uncertainties
        )
    )

    report.append(
        "\n### Evidence Gaps\n"
    )

    report.append(
        markdown_bullets(
            analysis_result.evidence_gaps
        )
    )

    # ------------------------------------------------
    # Cross-source relations
    # ------------------------------------------------

    report.append(
        "\n## Cross-Source Scientific Relationships\n"
    )

    report.append(
        cross_source_report.summary
    )

    for relation in (
        cross_source_report.relations
    ):
        report.append(
            "\n"
            f"- **{relation.topic}** — "
            f"`{relation.claim_id_a}` ↔ "
            f"`{relation.claim_id_b}` — "
            f"**{relation.relation}**: "
            f"{relation.explanation}"
        )

    # ------------------------------------------------
    # Audited decision
    # ------------------------------------------------

    report.append(
        "\n## Audited Technical Decision\n"
    )

    report.append(
        audited_decision.decision_summary
    )

    report.append(
        "\n### System Direction\n"
    )

    report.append(
        audited_decision
        .system_direction
        .statement
    )

    report.append(
        "\nEvidence: "
        + evidence_ids_text(
            audited_decision
            .system_direction
            .evidence_ids
        )
    )

    decision_sections = [
        (
            "Data Strategy",
            audited_decision.data_strategy
        ),
        (
            "Baseline Strategy",
            audited_decision.baseline_strategy
        ),
        (
            "Advanced Model Strategy",
            audited_decision.advanced_model_strategy
        ),
        (
            "Evaluation Strategy",
            audited_decision.evaluation_strategy
        ),
        (
            "Implementation Constraints",
            audited_decision.implementation_constraints
        )
    ]

    for section_title, statements in (
        decision_sections
    ):

        report.append(
            f"\n### {section_title}\n"
        )

        for item in statements:

            report.append(
                f"- {item.statement} "
                f"**[Evidence: "
                f"{evidence_ids_text(item.evidence_ids)}]**"
            )

    # ------------------------------------------------
    # Roadmap
    # ------------------------------------------------

    report.append(
        "\n## Audited Implementation Roadmap\n"
    )

    for step in audited_roadmap.steps:

        report.append(
            f"\n### {step.step_id} — "
            f"{step.phase}\n"
        )

        report.append(
            f"**Objective:** "
            f"{step.objective}\n"
        )

        report.append(
            f"**Basis:** "
            f"{step.basis_type}\n"
        )

        report.append(
            f"**Evidence:** "
            f"{evidence_ids_text(step.evidence_ids)}\n"
        )

        report.append(
            "**Actions:**\n"
        )

        report.append(
            markdown_bullets(
                step.actions
            )
        )

        report.append(
            "\n**Deliverable:**\n"
        )

        report.append(
            step.deliverable
        )

        report.append(
            "\n**Validation Criteria:**\n"
        )

        report.append(
            markdown_bullets(
                step.validation_criteria
            )
        )

        report.append(
            "\n**Dependencies:**\n"
        )

        report.append(
            markdown_bullets(
                step.dependencies
            )
        )

    # ------------------------------------------------
    # Open questions
    # ------------------------------------------------

    report.append(
        "\n## Unresolved Research Questions\n"
    )

    report.append(
        markdown_bullets(
            audited_roadmap
            .unresolved_research_questions
        )
    )

    report.append(
        "\n## Engineering Assumptions\n"
    )

    report.append(
        markdown_bullets(
            audited_roadmap
            .engineering_assumptions
        )
    )

    report.append(
        "\n## Final Implementation Target\n"
    )

    report.append(
        audited_roadmap.final_target
    )

    # ------------------------------------------------
    # Evidence registry
    # ------------------------------------------------

    report.append(
        "\n## Evidence Registry\n"
    )

    for item in evidence_store:

        report.append(
            f"\n### {item.evidence_id} — "
            f"{item.source_name}\n"
        )

        report.append(
            f"- Source type: "
            f"{item.source_type}"
        )

        report.append(
            f"- Role: {item.role}"
        )

        report.append(
            f"- Evidence status: "
            f"{item.evidence_status}"
        )

        if item.source_url:
            report.append(
                f"- Source URL: "
                f"{item.source_url}"
            )

        report.append(
            f"- Evidence: "
            f"{item.content}"
        )

        if item.details:

            report.append(
                "- Details:"
            )

            for detail in item.details:
                report.append(
                    f"  - {detail}"
                )

        report.append(
            f"- Provenance: "
            f"{item.provenance_note}"
        )

    # ------------------------------------------------
    # Audit trail
    # ------------------------------------------------

    decision_partial = sum(
        1
        for item in decision_audit.audits
        if item.status
        == "partially_supported"
    )

    decision_unsupported = sum(
        1
        for item in decision_audit.audits
        if item.status
        == "unsupported"
    )

    roadmap_partial = sum(
        1
        for item in roadmap_audit.audits
        if item.status
        == "partially_supported"
    )

    roadmap_unsupported = sum(
        1
        for item in roadmap_audit.audits
        if item.status
        == "unsupported"
    )

    report.append(
        "\n## Evidence Audit Trail\n"
    )

    report.append(
        f"- Decision statements revised after semantic audit: "
        f"{decision_partial}"
    )

    report.append(
        f"- Unsupported decision statements: "
        f"{decision_unsupported}"
    )

    report.append(
        f"- Roadmap steps requiring semantic revision: "
        f"{roadmap_partial}"
    )

    report.append(
        f"- Unsupported roadmap steps: "
        f"{roadmap_unsupported}"
    )

    report.append(
        f"- Final roadmap invalid evidence IDs: "
        f"{len(final_roadmap_validation['invalid_evidence_ids'])}"
    )

    report.append(
        f"- Final roadmap overreach flags: "
        f"{len(final_roadmap_validation['overreach_flags'])}"
    )

    report.append(
        "\n## Scope Note\n"
    )

    report.append(
        "This report distinguishes scientific claims, dataset metadata, "
        "repository documentation, official technical documentation, "
        "and engineering scaffolding. Evidence-backed recommendations "
        "should not be interpreted as proof that a particular model, "
        "dataset, or implementation will outperform alternatives without "
        "further empirical evaluation."
    )

    return "\n".join(report)

In [150]:
final_report = build_final_aura_report(
    question,
    analysis_result,
    cross_source_report,
    audited_decision_result,
    audited_roadmap_result,
    unified_evidence_store,
    decision_audit,
    roadmap_audit,
    final_roadmap_validation
)

print(final_report[:5000])

# AURA — Scientific & Technical Research Report

## Research Question

I want to build a solar flare prediction system.

Find relevant scientific papers, datasets, machine learning methods,
existing code implementations, and technical documentation.

Compare the approaches, identify their limitations,
and propose an implementation plan.

## Evidence Coverage

- Total unified evidence items: 32
- Paper: 21
- Dataset: 3
- Code: 5
- Documentation: 3

### Evidence Status

- verified_supported: 21
- metadata_supported: 3
- readme_supported: 5
- official_documentation_supported: 3

## Scientific Evidence Synthesis

The synthesis of the provided sources demonstrates that machine learning and deep learning are extensively applied in solar physics and space weather for tasks such as nowcasting and forecasting geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flares, coronal mass ejection propagation time, and solar wind speed. Techniques like those used in the FLARECAS

In [152]:
report_filename = (
    "AURA_solar_flare_clean_report.md"
)

with open(
    report_filename,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        final_report
    )

print(
    f"✅ Final report saved: "
    f"{report_filename}"
)

print(
    f"Report characters: "
    f"{len(final_report)}"
)

✅ Final report saved: AURA_solar_flare_clean_report.md
Report characters: 53532


In [154]:
print("=" * 70)
print("AURA CLEAN MVP — FINAL CHECKPOINT")
print("=" * 70)

print(
    f"Verified paper claims: "
    f"{len(trusted_claims)}"
)

print(
    f"Dataset families: "
    f"{len(dataset_families)}"
)

print(
    f"Code evidence sources: "
    f"{len(code_evidence_store)}"
)

print(
    f"Documentation sources: "
    f"{len(documentation_evidence_store)}"
)

print(
    f"Unified evidence items: "
    f"{len(unified_evidence_store)}"
)

print(
    f"Cross-source relations: "
    f"{len(cross_source_report.relations)}"
)

print(
    f"Roadmap steps: "
    f"{len(audited_roadmap_result.steps)}"
)

print(
    f"Roadmap overreach flags: "
    f"{len(final_roadmap_validation['overreach_flags'])}"
)

print()

print(
    "✅ AURA Clean MVP pipeline completed"
)

AURA CLEAN MVP — FINAL CHECKPOINT
Verified paper claims: 21
Dataset families: 3
Code evidence sources: 5
Documentation sources: 3
Unified evidence items: 32
Cross-source relations: 10
Roadmap steps: 6
Roadmap overreach flags: 0

✅ AURA Clean MVP pipeline completed
